# 05 · Graph memory cookbook
### Recipes from *Graph-based Agent Memory* (arXiv 2602.05665), for a banking customer context vault

**Level 400 to 500.** Prerequisite: notebooks 01 to 04, in particular notebook 02 §2.8, where the vault's relations became a graph whose edges inherit the policy and validity window of the record asserting them.

Yang et al.'s survey (2026) organises agent memory by a **life cycle**: *extraction* (raw data to memory), *storage* (how it is organised), *retrieval* (how it is found) and *evolution* (how it changes over time). It distinguishes **knowledge memory** (facts, rules) from **experience memory** (outcomes, lessons), and argues that a structural, graph-based store is the general case of which buffers and vector stores are degenerate versions. Its final section names seven open challenges. This notebook turns the parts that matter for a bank's customer context vault into twenty recipes, each with the same shape:

> **Priority** for this use case (🔴 high, 🟠 medium, ⚪ lower) · **Problem** in this use case's terms · **Sources and what each contributes**: every paper the recipe draws on, its title, and what it contributed, followed by what this project added that is not from any paper · **Recipe** working code on the vault · **Measure** what it bought · **Watch out** the pitfall for a regulated setting

Reference numbers in brackets are the survey's own reference list, and contributions are as the survey describes them. The primary papers were not all read in full, so read each line as "the idea, as the survey describes it".

| Life-cycle stage (paper section) | Recipes |
|---|---|
| Extraction (§IV) | 1 schema-guided triples · 2 mention time vs event time, with arithmetic in code |
| Storage (§V) | 3 bi-temporal edges · 4 a hierarchical summary tree · 5 commitments as hyperedges |
| Retrieval (§VI) | 6 routed traversal and the token budget · 7 a temporal query operator · 8 multi-round retrieval with a sufficiency check · 9 hybrid sources with an authority rule |
| Evolution (§VII) | 10 consolidation · 11 significance and forgetting · 12 inferred links, labelled · 13 experience memory with human approval · 14 active inquiry |
| Challenges (§X) and benchmarks (§VIII) | 15 intrinsic graph quality · 16 relational privacy · 17 graph poisoning · 18 multi-agent write conflicts · 19 benchmark map · 20 stability across reruns |

Every code cell opens with a **Sources in this cell** header naming the paper behind each part of that cell and what it contributes, and what this project added. The notebook ends with questions and answers to check your understanding. Runtime is about 15 minutes and roughly 150 model calls; every recipe runs on the same vault, built once below.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   Setup (this project): builds the customer context vault of notebooks 01 to 04, with graph memory (notebook 02 §2.8)
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
import json, math, re, sys, time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field

sys.path.insert(0, str(Path.cwd()))
import memlab
from memlab.config import build_memory, build_model
from memlab.evaluation import load_dataset, retrieval_sufficiency, judge, judge_vault_answer, answer_vault, run_parallel, with_retry
from memlab.vault import (load_vault_dataset, build_vault, vault_rows, view, render, normalise, raw_sources, write_record,
                          write_playbook, Interaction)
from memlab.access import MemoryService, principals_from, authorize, vault_tools, route_layers, BANK_AGENT_PROMPT, CUSTOMER_AGENT_PROMPT
from memlab.graph import GraphIndex, Edge, canonical
from strands import Agent, tool

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 180)
dataset, vd = load_dataset(), load_vault_dataset()
TODAY, directory = vd["today"], vd["party_directory"]
principals = principals_from(vd["principals"])
alex, vrm = principals["banker_alex"], principals["vrm_priya"]
interactions = {i.source_ref: i for i in (normalise(r) for r in raw_sources(dataset, vd))}

started = time.perf_counter()
vault = build_memory("nb05_vault", fresh=True)
built = build_vault(vault, dataset, vd)
service = MemoryService(vault, today=TODAY, directory=directory)
service.enable_graph()
graph = service.graph
rows = view(vault_rows(vault, "cust_priya"), today=TODAY)
print(f"vault built in {time.perf_counter() - started:.0f}s: {len(rows)} records for Priya, graph of {len(graph.edges)} edges "
      f"between {len(graph.adjacent)} entities across {len({e.customer_id for e in graph.edges})} customers")

Model us.anthropic.claude-haiku-4-5-20251001-v1:0 may not be available in region us-east-1
The 'faiss' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).


vault built in 130s: 63 records for Priya, graph of 18 edges between 9 entities across 3 customers


### The paper's taxonomy, mapped onto the vault

| The survey's distinction | In this vault |
|---|---|
| Short-term vs long-term memory | Working and session memory (notebook 01, Parts 1 to 3) vs the vault |
| Knowledge vs experience memory | Semantic, preference and system-of-record records vs episodic records, commitments, corrections and playbooks |
| Non-structural vs structural memory | mem0's flat facts (the `flat_mem0` baseline) vs typed records with slots, links and the graph |
| Semantic, procedural, associative, working, episodic, sentiment memory | `layer` = semantic, procedural, episodic; associative = graph edges and inferred links (recipe 12); sentiment = interpretations (`kind=derived`) |
| Write, read, update, delete | `ingest`, `MemoryService.search`, supersession and closure, lineage-based deletion (notebook 04) |

### Where to focus first

Every recipe opens with a priority marker for this use case. The high-importance recipes, in the order to read them:

| Recipe | Why it matters for this use case |
|---|---|
| 17 | Memory poisoning: one customer statement can rewire the graph and link unrelated customers unless unverified structure is quarantined. |
| 16 | Graph structure can reveal one customer's relationships in another's channel without any record being read. |
| 18 | The VRM and the banker write to the same customer's memory; order-dependent results are nondeterministic in a queue. |
| 9 | Systems of record and customer statements disagree in real data; presenting only one value misleads the banker. |
| 5 | Commitments are the facts a bank is most accountable for; keeping each promise whole keeps due dates and owners right. |
| 2 | Commitments, due dates and 'when did it happen' questions are core to a bank, and dates are where extraction goes wrong most. |
| 3 | Audit and complaints ask what the bank knew when it acted; without transaction time the vault cannot answer. |
| 1 | Every graph edge comes from this step; inconsistent relation names silently split relationships and break traversal. |
| 20 | A release decision taken on one run is taken on noise; every gate result depends on this. |

Medium: 4, 6, 7, 8, 10, 12, 13, 14, 15. Lower: 11, 19.

# Part I · Extraction

## Recipe 1 · Schema-guided triples

> 🔴 **High importance for this use case.** Every graph edge comes from this step; inconsistent relation names silently split relationships and break traversal.

**Problem.** The graph is only as good as the relations the extractor happens to write. Left free, a model names the same relationship five ways (`accountant_of`, `is_accountant_for`, `handles_bas_for`), and traversal silently splits one relation into several.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[20] Zeng et al.**, *On the structural memory of LLM agents* (2024), and **[21] Liang et al.**, *PersonaAgent with GraphRAG* (2025): cited in §IV for prompting a model to extract (entity, relation, entity) triples from text. *Used here:* the extraction step itself.
- **[30] Anokhin et al.**, *AriGraph: Learning knowledge graph world models with episodic memory for LLM agents* (2024): an LLM parses observations into triples to build the memory graph (§V). *Used here:* the idea of a triple-based graph built from interaction text.
- **§X, dynamic schema learning** (citing **[11] Shang et al.**, *AgentSquare*, 2025, on domain-specific schemas needing re-engineering): an open challenge, that agents should learn relevant entity and relation types themselves. *Used here:* the motivation for a governed vocabulary that grows through review.
- *This project, not from a paper:* the controlled predicate list, the explicit `OTHER`, the schema-gap review queue, and the free-vs-guided comparison.

**Recipe.** Give the extractor a controlled predicate vocabulary, keep what it cannot map as *schema-gap candidates* for a person to review, and compare with free extraction on the same three interactions.

In [2]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [20] Zeng et al. / [21] Liang et al.  ->  prompting an LLM to extract (entity, relation, entity) triples: extract_triples()
#   [30] AriGraph                       ->  triples from interaction text as the memory graph
#   §X dynamic schema learning          ->  motivates PREDICATES + OTHER; the gap review is this project's governed form
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
PREDICATES = {
    "director_of": "a person is a director of a company", "owns": "an entity owns an asset or premises",
    "operates_from": "a business operates from premises", "tenant_of": "a business rents premises",
    "related_entity_of": "two companies are related", "accountant_of": "an external accountant acts for a business",
    "relationship_manager_of": "a banker manages a customer", "holds": "an entity holds a bank product or facility",
    "secured_by": "a facility is secured by an asset", "guarantees": "a person guarantees a facility",
    "plans": "a business plans a site, purchase or change", "employs": "a business employs a person or role",
}

class Triple(BaseModel):
    subject: str
    predicate: str
    object: str

class Triples(BaseModel):
    triples: list[Triple]

def extract_triples(text: str, schema: dict | None = None) -> list[Triple]:
    rule = ("Use ONLY these predicates; if a relationship fits none, use predicate OTHER:" + json.dumps(schema)) if schema else \
           "Use short snake_case predicates of your choice."
    agent = Agent(model=build_model(temperature=0.0), callback_handler=None,
                  system_prompt="Extract relationships between named people, businesses, assets and bank products as triples. " + rule)
    return with_retry(lambda: agent(text, structured_output_model=Triples)).structured_output.triples

sources = ["contact_centre:CALL-88213", "email:EM-77164", "vrm:VRM-310554"]
free = run_parallel(lambda ref: extract_triples(interactions[ref].text), sources)
guided = run_parallel(lambda ref: extract_triples(interactions[ref].text, PREDICATES), sources)

free_preds = Counter(t.predicate for ts in free for t in ts)
guided_preds = Counter(t.predicate for ts in guided for t in ts)
print("free extraction, predicates used:  ", dict(free_preds))
print("schema-guided, predicates used:   ", dict(guided_preds))
gaps = [(ref, t) for ref, ts in zip(sources, guided) for t in ts if t.predicate not in PREDICATES]
print(f"\nschema-gap candidates for review ({len(gaps)}):")
for ref, t in gaps:
    print(f"  {ref:<28} {t.subject} --?--> {t.object}")

free extraction, predicates used:   {'is_customer_of': 1, 'is_banker_for': 1, 'has_accountant': 1, 'manages_bas_for': 1, 'is_for': 1, 'credit_amount': 1, 'credit_date': 1, 'credit_to': 1, 'owns_merchant_account': 1, 'lease_signed_date': 1, 'located_at': 1, 'opening_date': 1, 'operates': 1, 'required_by_date': 4, 'send_date': 1, 'send_to': 1, 'to_be_copied_on': 3, 'works_at': 2, 'email': 2, 'sent_email_to': 1, 'role': 1, 'works_for': 1, 'belongs_to': 1, 'original_opening_date': 1, 'new_opening_date': 1, 'delay_reason': 1, 'needed_for': 2, 'should_be_sent_to': 2, 'operates_depot': 1, 'owns_fleet': 1, 'purchased_vehicles': 1, 'purchase_date': 1, 'planned_hiring': 1, 'weekly_wage': 1, 'monthly_cost': 1, 'pending_credit': 1, 'due_date': 1, 'bank_contact': 1}
schema-guided, predicates used:    {'relationship_manager_of': 2, 'holds': 6, 'tenant_of': 1, 'operates_from': 3, 'accountant_of': 3, 'plans': 3, 'owns': 1, 'OTHER': 1}

schema-gap candidates for review (1):
  vrm:VRM-310554            

**Measure.** Count distinct predicates per relationship type: free extraction typically invents several names for one relation, the schema-guided run uses the vocabulary and surfaces the rest as gaps. **Watch out.** A schema that is too small forces wrong labels (a model will squeeze "manages the BAS" into `relationship_manager_of`). Keep an explicit `OTHER`, review the gaps weekly, and version the predicate list like the extraction policy (notebook 04 §2.6). That review loop is the practical, governed form of the paper's dynamic schema learning.

## Recipe 2 · Mention time, event time, and arithmetic in code

> 🔴 **High importance for this use case.** Commitments, due dates and 'when did it happen' questions are core to a bank, and dates are where extraction goes wrong most.

**Problem.** On 4 September Priya says "we signed the lease on Wednesday" and "the depot opens Monday the 2nd of November". The conversation date (mention time) and the dates it refers to (event time) are different, and temporal questions ("how long from the fault report to the fix?") go wrong when a model does date arithmetic in prose.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[37] Ge et al.**, *TReMu: Towards neuro-symbolic temporal reasoning for LLM-agents with memory in multi-session dialogues* (Findings of ACL 2025): decouples the mention time (session timestamp) from the inferred event time, organises memory as timeline summaries indexed by absolute time, and has the model generate Python for date arithmetic (§V). *Used here:* the mention-date / event-date split, and doing the arithmetic in code.
- **[38] Tan et al.**, *MemoTime: Memory-augmented temporal knowledge graph enhanced LLM reasoning* (arXiv 2510.13614, 2025): enforces temporal monotonicity so reasoning never uses an effect before its cause (§V). *Used here:* not in this recipe's cells; implemented as `check_temporal_order` and `monotonic` in `memlab/techniques.py`.
- *This project, not from a paper:* hand-written arithmetic functions in place of model-written code (TReMu's version), the business-day rule, and the `date_interval` tool.

**Recipe.** Extract dated events with both times, then answer interval questions with a deterministic function, including business days.

In [3]:
# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [37] TReMu  ->  mention_date vs event_date (TimedEvent) and date arithmetic in code (add_business_days, days_between, date_interval)
#   This project ->  hand-written arithmetic functions and the business-day rule
# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
class TimedEvent(BaseModel):
    event: str
    expression: str = Field(description="the words used, e.g. 'on Wednesday'")
    event_date: str | None = Field(description="YYYY-MM-DD the event happens or happened, resolved against the mention date")
    mention_date: str

class Timeline(BaseModel):
    events: list[TimedEvent]

call = interactions["contact_centre:CALL-88213"]
agent = Agent(model=build_model(temperature=0.0), callback_handler=None,
              system_prompt="List every dated event in the conversation. Resolve relative dates against the mention date given.")
timeline = with_retry(lambda: agent(f"Mention date: {call.occurred_on} (a Friday)\n\n{call.text}", structured_output_model=Timeline)).structured_output
for e in timeline.events:
    print(f"  mentioned {e.mention_date}  event {e.event_date}  '{e.expression}'  {e.event[:70]}")

def add_business_days(start: str, n: int, holidays: frozenset = frozenset()) -> str:
    d = date.fromisoformat(start)
    while n:
        d += timedelta(days=1)
        if d.weekday() < 5 and d.isoformat() not in holidays:
            n -= 1
    return d.isoformat()

days_between = lambda a, b: (date.fromisoformat(b) - date.fromisoformat(a)).days
print("\nwaiver promised 2 July 'within 5 business days' -> due", add_business_days("2026-07-02", 5))
print("fault reported 10 June -> technician fixed it 3 August:", days_between("2026-06-10", "2026-08-03"), "days")

@tool
def date_interval(start: str, end: str) -> str:
    '''Days between two dates (YYYY-MM-DD), and the weekday of each.

    Args:
        start: first date
        end: second date
    '''
    s, e = date.fromisoformat(start), date.fromisoformat(end)
    return f"{(e - s).days} days ({s:%A} to {e:%A})"

q12 = next(q for q in dataset["questions"] if q["id"] == "q12")
with_tool = Agent(model=build_model(), callback_handler=None, tools=vault_tools(service, alex, "cust_priya") + [date_interval],
                  system_prompt=BANK_AGENT_PROMPT.format(today=TODAY) + " Use date_interval for any arithmetic on dates.")
print(f"\nQ: {q12['question']}\nA: {str(with_tool(q12['question'])).strip()[:300]}\n(gold: {q12['gold_answer']})")

  mentioned 2026-09-04  event 2026-07-01  'in July'  Merchant team moved to a new platform
  mentioned 2026-09-04  event 2026-09-11  'by Friday the 11th of September'  $29 terminal fee waiver credited to merchant account
  mentioned 2026-09-04  event 2026-09-02  'on Wednesday'  Lease signed on the Beresfield site
  mentioned 2026-09-04  event 2026-11-02  'on Monday the 2nd of November'  Newcastle depot opens
  mentioned 2026-09-04  event 2026-09-25  'by Friday the 25th of September'  Newcastle merchant facility application sent

waiver promised 2 July 'within 5 business days' -> due 2026-07-09
fault reported 10 June -> technician fixed it 3 August: 54 days

Q: How long did it take from Priya first reporting the terminal problem to the technician finally swapping in a working unit?
A: **It took 54 days** from when Priya first reported the terminal problem on 2026-06-10 (Wednesday) to when the technician installed the replacement working unit on 2026-08-03 (Monday).

During this period, 

The cell above resolves dates with a model and does arithmetic in hand-written functions. The next cell adds the three pieces the two papers contribute beyond that, using the shared utilities in `memlab/techniques.py`:

1. **Rule-based grounding with a granularity** (TReMu's decoupling, done deterministically): "in July" stays a *month* instead of becoming 1 July, a weekday that does not match its date is flagged, and anything that cannot be grounded safely goes to review instead of being guessed.
2. **Ordering checks** (MemoTime's monotonicity, applied when records are written): no item closed before it was opened, no promise due before it was made.
3. **The model chooses the date operation, code computes it** (TReMu's neuro-symbolic step, in a governed form): scored on notebook 03's four temporal questions.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [37] TReMu    ->  (1) ground_time: decouple mention time from event time; (3) answer_temporal: model picks the operation, code computes
#   [38] MemoTime ->  (2) check_temporal_order / monotonic: temporal monotonicity
#   This project  ->  granularity field, 'return None rather than guess', fixed operation set instead of generated code
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
from memlab.techniques import ground_time, answer_temporal, check_temporal_order, monotonic

# (1) [37] TReMu: ground each expression against its mention date, with a granularity, deterministically
grounded = []
for e in timeline.events:
    tense = "future" if (e.event_date or "") > e.mention_date else "past"
    g = ground_time(e.expression, e.mention_date, tense=tense)
    grounded.append({"expression": e.expression, "model's event_date": e.event_date,
                     "rule-based": "not grounded: review" if g is None else (g.start if g.start == g.end else f"{g.start} .. {g.end}"),
                     "granularity": g.granularity if g else "-", "agree": bool(g) and g.start <= (e.event_date or "") <= g.end,
                     "note": g.note if g else ""})
print(pd.DataFrame(grounded).to_string(index=False))

# (2) [38] MemoTime: temporal monotonicity, checked on every record of the vault at write time
violations = [(r["text"][:80], v) for r in rows for v in check_temporal_order(r, rows)]
print(f"\nordering violations across {len(rows)} records: {len(violations)}")
for text, v in violations[:5]:
    print(f"  {v}  <- {text}")
print("example chain check:", monotonic([("2026-07-02", "waiver promised"), ("2026-09-04", "re-promised by 11 Sep"),
                                         ("2026-09-14", "complaint lodged"), ("2026-09-16", "credit posted")]) or "in a possible order")

# (3) [37] TReMu: the model picks ONE date operation from a fixed set; the arithmetic is done in code.
#     Two timelines: indexed by MENTION date (when the record was written) and by EVENT date (the dates the text states),
#     which is TReMu's "timeline summaries indexed by inferred absolute time".
from memlab.techniques import MONTHS
DATE_IN_TEXT = re.compile(r"\b(\d{4}-\d{2}-\d{2})\b|\b(\d{1,2}) (" + "|".join(m.title() for m in MONTHS) + r") (\d{4})\b")
def event_dates(r):
    found = [m[1] or f"{m[4]}-{MONTHS.index(m[3].lower()) + 1:02d}-{int(m[2]):02d}" for m in DATE_IN_TEXT.finditer(r["text"])]
    return found or [r["observed_on"]]
history_rows = [r for r in rows if r.get("layer") in ("episodic", "commitment")]
by_mention = sorted({(r["observed_on"], r["text"][:120]) for r in history_rows})
by_event = sorted({(d, r["text"][:120]) for r in history_rows for d in event_dates(r)}
                  | {(r["due_date"], "DUE: " + r["text"][:110]) for r in history_rows if r.get("due_date")})   # a due date is an event date too

def correct(qid, o):
    used = {o["a"], o.get("b"), str(o["result"])}
    return {"q10": o["result"] == 9, "q12": o["result"] == 54,
            "q11": (o["operation"] == "earlier_of" and o["result"] == "2026-07-21") or {o["a"], o.get("b")} == {"2026-07-21", "2026-08-07"},
            "q13": "2026-07-09" in used}[qid]
EXPECTED = {"q10": "9 days (10 Jun -> 19 Jun)", "q11": "refund 21 Jul before settlement 7 Aug", "q12": "54 days (10 Jun -> 3 Aug)", "q13": "due 9 Jul"}
temporal_qs = [q for q in dataset["questions"] if q["id"] in EXPECTED]
results = []
for name, timeline_ in (("by mention date", by_mention), ("by event date (TReMu)", by_event)):
    for q, o in zip(temporal_qs, run_parallel(lambda q: answer_temporal(q["question"], timeline_), temporal_qs)):
        results.append({"timeline": name, "question": q["id"], "operation": o["operation"], "a": o["a"], "b": o.get("b"),
                        "result": o["result"], "expected": EXPECTED[q["id"]], "correct": correct(q["id"], o)})
results = pd.DataFrame(results)
print("\n" + results.to_string(index=False))
print("\ncorrect by timeline:", results.groupby("timeline", sort=False).correct.sum().to_dict(), f"of {len(temporal_qs)}")

                     expression model's event_date               rule-based granularity  agree note
                        in July         2026-07-01 2026-07-01 .. 2026-07-31       month   True     
by Friday the 11th of September         2026-09-11               2026-09-11         day   True     
                   on Wednesday         2026-09-02               2026-09-02         day   True     
  on Monday the 2nd of November         2026-11-02               2026-11-02         day   True     
by Friday the 25th of September         2026-09-25               2026-09-25         day   True     

ordering violations across 63 records: 0
example chain check: in a possible order

             timeline question    operation          a          b  result                              expected  correct
      by mention date      q10 days_between 2026-06-11 2026-06-19       8             9 days (10 Jun -> 19 Jun)    False
      by mention date      q11 days_between 2026-07-21 2026-08-14      24 

**Measure.** Ordering violations are a property of this build's extraction, so the count changes between runs. A zero means every record was opened before it was closed and promised before it was due. A nonzero count lists records whose supersession or closure points backwards in time (for example a later report closed by an earlier one); MemoTime's rule would hold each of those for review instead of letting the closure stand. The interval and the business-day due date are now exact; the model's job is only to find the dates. In the grounding table, read the granularity column ("in July" is a month, not a day) and any row the rules could not ground: those go to review rather than into memory with false precision. In the last table, a wrong row is a wrong *choice of dates or operation* by the model, never wrong arithmetic, which makes it easy to diagnose. Compare the two timelines: indexed by mention date, the model sees "settled on 7 August" filed under 14 August (the call in which Priya reported it) and computes from the wrong date; indexed by the event dates the records state, the same question gets the right inputs. That is TReMu's point about timelines: index events by when they happened, not by when they were said. **Watch out.** Business days need a holiday calendar (NSW public holidays, bank holidays), and "within five business days" is a policy definition that belongs to the bank, not the model.

# Part II · Storage

## Recipe 3 · Bi-temporal edges: true on a date vs known on a date

> 🔴 **High importance for this use case.** Audit and complaints ask what the bank knew when it acted; without transaction time the vault cannot answer.

**Problem.** Arjun resigned as a director on 31 August, but suppose the KYC system was updated only on 10 September. A banker asks: *who did our records show as directors on 5 September?* The valid-time view says Priya alone, which is true of the world but not what the bank knew. Audit and complaints questions are about what the bank knew.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[18] Rasmussen et al.**, *Zep: a temporal knowledge graph architecture for agent memory* (arXiv 2501.13956, 2025); Graphiti is its open-source graph engine: a bi-temporal model that separates valid time from transaction time, tracks when each fact was created and expired, and invalidates rather than overwrites (§V). *Used here:* the two time axes and the "true on / known on" query.
- *This project, not from a paper:* the KYC keying-lag scenario, and the rule that a supersession not yet recorded on the known-on date does not end an edge.

**Recipe.** Add a recorded-at time to every edge (here, a KYC keying lag for the resignation record; in production, the write timestamp), and query on both axes.

In [5]:
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [18] Zep / Graphiti  ->  bi-temporal facts: valid time (valid_on) and transaction time (known_on); invalidate, never overwrite
#   This project        ->  the KYC keying-lag scenario; an unrecorded supersession does not end an edge
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
RECORDED_AT = {"kyc:CIF-100231:directors-2026-08": "2026-09-10"}      # keyed in ten days after the resignation
recorded = lambda record: RECORDED_AT.get(record["source_ref"], record.get("observed_on") or "")

def directors(valid_on: str, known_on: str) -> list[str]:
    '''Directors of Sharma Logistics true on `valid_on`, as the bank's records stood on `known_on`.'''
    found = set()
    for e in graph.edges:
        rec = graph.records[e.record_id]
        if e.predicate != "director_of" or e.object != "sharma logistics" or recorded(rec) > known_on:
            continue
        superseder = graph.records.get(rec.get("superseded_by") or "")
        ends = rec.get("superseded_on") if superseder and recorded(superseder) <= known_on else None   # unknown supersession = still current
        if e.valid_from <= valid_on and (ends is None or ends > valid_on):
            found.add(graph.labels[e.subject])
    return sorted(found)

grid = [(v, k) for v in ("2026-08-01", "2026-09-05") for k in ("2026-09-05", TODAY)]
print(pd.DataFrame([{"true on (valid time)": v, "as known on (transaction time)": k, "directors": ", ".join(directors(v, k))} for v, k in grid]).to_string(index=False))

true on (valid time) as known on (transaction time)                  directors
          2026-08-01                     2026-09-05 Arjun Sharma, Priya Sharma
          2026-08-01                     2026-09-27 Arjun Sharma, Priya Sharma
          2026-09-05                     2026-09-05 Arjun Sharma, Priya Sharma
          2026-09-05                     2026-09-27               Priya Sharma


**Measure.** The row "true on 5 September, as known on 5 September" says Priya and Arjun: the bank's records were stale, and that is the correct answer to an audit question. The same date "as known today" says Priya alone. **Watch out.** Transaction time must be the write time in the system of record or the vault, never re-derived later; and a supersession that was not yet recorded must not end an edge in a known-on query.

## Recipe 4 · A hierarchical summary tree

> 🟠 **Medium importance for this use case.** Needed once histories run to years; one summary level is enough for the pilot scope.

**Problem.** Years of episodic history will not fit in a Context Pack, and a flat list of events hides the story ("three terminals, two failures, fixed on 3 August").

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **MemTree** (described in §V; no citation number in the survey text): routes new information through a hierarchy, clustering it under existing nodes or creating branches, and recursively updates every ancestor's summary. *Used here:* topic nodes with summaries under a root summary.
- **[34] Patel and Patel**, *ENGRAM: Effective, lightweight memory orchestration for conversational agents* (2025): cited for the two construction processes, semantic clustering for organisation and recursive summarisation for abstraction (§V). The survey also notes ENGRAM's finding that a basic typed memory can match complex systems. *Used here:* clustering plus summarisation in one model call.
- **[35] Wu et al.**, *SGMem: Sentence graph memory for long-term conversational agents* (2025): a sentence-level tree linking utterances by sequence and reference. *Used here:* not implemented; the session-memory variant of the same idea.
- *This project, not from a paper:* the coverage, duplicate and compression checks, drill-down, and the rule that a summary inherits its members' most restrictive audience.

**Recipe.** Group current episodic records and commitments into topic nodes with summaries and member ids, add a root summary, and drill down on demand. Check coverage (every record placed) and the compression ratio.

In [6]:
# ──────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   MemTree (§V)  ->  topic nodes, root summary, recursive summaries
#   [34] ENGRAM   ->  semantic clustering + recursive summarisation in one step
#   This project  ->  coverage / duplicate / compression checks and drill_down()
# ──────────────────────────────────────────────────────────────────────────────
class Topic(BaseModel):
    name: str
    summary: str = Field(description="two sentences with dates")
    member_ids: list[str] = Field(description="8-character ids of the records in this topic")

class Tree(BaseModel):
    root_summary: str = Field(description="three sentences: the relationship as a whole")
    topics: list[Topic]

leaves = [r for r in rows if r.get("layer") in ("episodic", "commitment") and r["current"]]
writer = Agent(model=build_model(temperature=0.0), callback_handler=None,
               system_prompt="Organise a bank customer's history into 3 to 6 topics. Every record id must appear in exactly one topic. Cite ids.")
tree = with_retry(lambda: writer("\n".join(render(r) for r in leaves), structured_output_model=Tree)).structured_output

by_short = {r["id"][:8]: r for r in leaves}
placed = [i[:8] for t in tree.topics for i in t.member_ids if i[:8] in by_short]
tokens = lambda texts: sum(len(t) for t in texts) / 4
print(f"root: {tree.root_summary}\n")
for t in tree.topics:
    print(f"[{t.name}] ({len(t.member_ids)} records) {t.summary}")
print(f"\ncoverage: {len(set(placed))} of {len(leaves)} records placed; duplicates {len(placed) - len(set(placed))}; "
      f"compression: {tokens([tree.root_summary] + [t.summary for t in tree.topics]):.0f} tokens for root and topics vs "
      f"{tokens([r['text'] for r in leaves]):.0f} for the leaves")

def drill_down(question: str) -> list[str]:
    words = set(re.findall(r"[a-z]{4,}", question.lower()))
    best = max(tree.topics, key=lambda t: sum(w in (t.name + " " + t.summary).lower() for w in words))
    return [best.name] + [by_short[i[:8]]["text"] for i in best.member_ids if i[:8] in by_short]

print("\ndrill-down for 'what went wrong with the terminals?':")
for line in drill_down("what went wrong with the terminals?")[:5]:
    print("  " + line[:120])

root: Sharma Logistics Pty Ltd's relationship with the bank spans from June 2026 to September 2026, involving merchant EFTPOS facility issues, equipment financing, and expansion planning. The customer experienced service failures with terminal faults and fee processing delays, but the bank has committed to supporting their Newcastle depot expansion with merchant facilities by November 2026. Key contact is Priya Sharma, with relationship manager Alex Nguyen and accountant Deepak Rao involved.

[EFTPOS Terminal Fault and Resolution (June-August 2026)] (11 records) Sharma Logistics Pty Ltd's Parramatta depot EFTPOS terminal experienced connection failures from 2026-06-04, logged as fault M-2287 on 2026-06-10. The bank arranged a technician visit on 2026-08-03 with a replacement unit installed, which has been working reliably since.
[Terminal Rental Fee Waiver Dispute (June-September 2026)] (13 records) The bank failed to process a $29 June 2026 terminal rental fee waiver by the committed 

**Measure.** Coverage must be 100% with no duplicates, or the tree silently forgets; compression is the reason to have it. **Watch out.** A summary is derived memory: it inherits the most restrictive audience of its members (notebook 04 §2.5), and it must be recomputed when a member changes, which is why the survey stresses *recursive update of ancestor summaries* rather than appending.

## Recipe 5 · Commitments as hyperedges

> 🔴 **High importance for this use case.** Commitments are the facts a bank is most accountable for; keeping each promise whole keeps due dates and owners right.

**Problem.** A commitment is not a pair. "The bank will send Priya and Deepak the Newcastle merchant application by 25 September (re-raised from W-5530)" binds a bank, a customer, an accountant, an action, a due date and a reference. Split into triples, the fact that these belong to *one* promise is lost.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[39] Luo et al.**, *HyperGraphRAG: Retrieval-augmented generation via hypergraph-structured knowledge representation* (arXiv 2503.21322, 2025): a text fragment and all its entities form one hyperedge, enabling retrieval through any member (§V). *Used here:* one hyperedge per commitment.
- **[40] Huang et al.**, *HyperG: Hypergraph-enhanced LLMs for structured knowledge* (SIGIR 2025): hyperedges over rows, columns and tables of tabular data. *Used here:* not implemented; relevant if commitments come from tabular systems.
- *This project, not from a paper:* building hyperedges from typed commitment records, adding reference numbers as members, and the lookup by any member.

**Recipe.** Build a hyperedge per commitment from the typed record (members = bank, customer, named entities, references), then query by any member.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [39] HyperGraphRAG  ->  one hyperedge per commitment holding all its entities; lookup by any member
#   This project       ->  references as members, one canonical form per reference
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
REF = re.compile(r"\b[A-Z]{1,3}-\d{3,6}\b")
hyperedges = []
for r in rows:
    if r.get("layer") != "commitment":
        continue
    # a reference keeps its hyphen whether it arrives as an entity or in the text, so C-9912 is one member, not two
    members = {"the bank", "sharma logistics"} | {e.lower() if REF.fullmatch(e) else canonical(e) for e in (r.get("entities") or [])} \
              | {m.lower() for m in REF.findall(r["text"])}
    hyperedges.append({"id": r["id"][:8], "members": members, "due": r.get("due_date"), "state": r.get("state"), "text": r["text"]})

def promises_involving(member: str) -> list[dict]:
    m = canonical(member) if not REF.fullmatch(member) else member.lower()
    return [h for h in hyperedges if m in h["members"]]

top_ref = Counter(m for h in hyperedges for m in h["members"] if REF.fullmatch(m.upper())).most_common(1)
for member in ["Deepak Rao"] + [r.upper() for r, _ in top_ref]:
    found = promises_involving(member)
    print(f"commitments involving {member}:" + ("" if found else " none in this build"))
    for h in found:
        print(f"  [{h['state']:<7} due {h['due']}] {h['text'][:110]}")
print(f"\n{len(hyperedges)} commitments as hyperedges; mean members per commitment {sum(len(h['members']) for h in hyperedges) / max(1, len(hyperedges)):.1f}")

commitments involving Deepak Rao: none in this build
commitments involving C-9912:
  [open    due 2026-10-05] The bank committed to resolve complaint C-9912 and respond to Priya Sharma of Sharma Logistics Pty Ltd in writ

12 commitments as hyperedges; mean members per commitment 3.2


The flat hyperedges above have two weaknesses you can see in the output: members carry **no roles** (who promised, who benefits, who is copied), and participants are **missing** whenever the extractor did not list them as entities (Priya and Deepak on the overdue Newcastle promise). The next cell builds **role-labelled hyperedges**: one model call per open commitment reads the commitment and the customer's standing instructions (for example "copy my accountant on loan and merchant paperwork") and returns each role with the source it came from. It also says whether the record is really a bank promise to the customer, which filters out internal suggestions. In a property graph this is the *reified* form: the promise becomes a node, and each role is a labelled edge to it.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [39] HyperGraphRAG  ->  one hyperedge per commitment holding all its entities; lookup by any member
#   This project       ->  references as members, one canonical form per reference
# ─────────────────────────────────────────────────────────────────────────────────────────────────────
ROLES = ("promisor", "beneficiary", "on_behalf_of", "object", "copy_to", "reference", "site")

class RoleValue(BaseModel):
    role: Literal["promisor", "beneficiary", "on_behalf_of", "object", "copy_to", "reference", "site"]
    value: str
    source: str = Field(description="the source reference of the line this role comes from")

class RoleEdge(BaseModel):
    bank_promise_to_customer: bool = Field(description="False for an internal suggestion, a customer request, or an AI agent's note")
    roles: list[RoleValue]

# standing instructions: the extractor types "copy Deepak on loan paperwork" as a preference in some builds and a semantic fact in others,
# so pass both layers and let the labeller decide what applies (it cites the source, so a reviewer can check)
standing = [r for r in rows if r["current"] and r.get("layer") in ("preference", "semantic") and r.get("source_system") not in ("kyc", "crm", "pds")]
open_items = [r for r in rows if r.get("layer") == "commitment" and r.get("state") in ("open", "overdue")]
labeller = lambda: Agent(model=build_model(temperature=0.0), callback_handler=None,
                         system_prompt="Label the participants of one bank commitment by role. Use the customer's standing instructions only "
                                       "where they clearly apply to this commitment (for example who to copy on paperwork). Cite the source of each role.")
def label(r):
    prompt = (f"Commitment [{r['source_ref']}], due {r.get('due_date')}, asserted by {r.get('asserted_by')}: {r['text']}\n\nStanding instructions:\n"
              + "\n".join(f"[{x['source_ref']}] {x['text']}" for x in standing))
    return with_retry(lambda: labeller()(prompt, structured_output_model=RoleEdge)).structured_output

labelled = run_parallel(label, open_items)
SOURCE_REF = re.compile(r"[a-z_]+:[A-Za-z0-9:\-]+")
ref_of = lambda text: (SOURCE_REF.search(text) or [text])[0]           # normalise "Commitment [contact_centre:CALL-88213]" to the reference
for r, e in zip(open_items, labelled):
    print(json.dumps({"id": r["id"][:8], "bank_promise_to_customer": e.bank_promise_to_customer, "due": r.get("due_date"), "state": r.get("state"),
                      "roles": {role: [(v.value, ref_of(v.source)) for v in e.roles if v.role == role] for role in ROLES if any(v.role == role for v in e.roles)},
                      "text": r["text"][:100]}, indent=1))

promises = [(r, e) for r, e in zip(open_items, labelled) if e.bank_promise_to_customer]
copied = [(r, e) for r, e in promises if any(v.role == "copy_to" and "deepak" in v.value.lower() for v in e.roles)]
print(f"\n{len(promises)} of {len(open_items)} open 'commitments' are bank promises to the customer; "
      f"promises that require copying Deepak Rao: {[(r['id'][:8], r.get('due_date'), r.get('state')) for r, _ in copied]}")
print("flat hyperedge lookup for Deepak Rao found:", [h["id"] for h in promises_involving("Deepak Rao")])

{
 "id": "6945ddbf",
 "bank_promise_to_customer": true,
 "due": "2026-10-05",
 "state": "open",
 "roles": {
  "promisor": [
   [
    "the bank",
    "commitment assertion"
   ]
  ],
  "beneficiary": [
   [
    "Priya Sharma of Sharma Logistics Pty Ltd",
    "commitment assertion"
   ]
  ],
  "copy_to": [
   [
    "Deepak Rao (accountant)",
    "contact_centre:CALL-88213"
   ]
  ]
 },
 "text": "The bank committed to resolve complaint C-9912 and respond to Priya Sharma of Sharma Logistics Pty L"
}
{
 "id": "c5f1a311",
 "bank_promise_to_customer": true,
 "due": "2026-09-25",
 "state": "overdue",
 "roles": {
  "promisor": [
   [
    "The bank",
    "Commitment assertion"
   ]
  ],
  "beneficiary": [
   [
    "Priya Sharma of Sharma Logistics Pty Ltd",
    "Commitment assertion"
   ]
  ],
  "object": [
   [
    "Newcastle merchant facility application",
    "Commitment assertion"
   ]
  ],
  "copy_to": [
   [
    "Deepak Rao (accountant)",
    "contact_centre:CALL-88213"
   ]
  ]
 },
 "text

**Measure.** One lookup by any participant returns the whole promise with its due date and state. Compare the last two lines: the role-labelled version should find the overdue Newcastle promise as one that requires copying Deepak (from his standing instruction), which the flat lookup misses, and should drop records that are not bank promises to the customer. **Watch out** for over-application: a model given a standing instruction ("copy my accountant on loan and merchant paperwork") may attach it to promises it does not cover, such as a promised phone call. That is why every role carries the source it came from: a role drawn from a standing instruction rather than the commitment itself is the one a reviewer should check, and a production rule can restrict `copy_to` to commitments whose object is paperwork. **Watch out.** Members extracted from text inherit the extractor's naming; canonicalise (recipe 1) or a hyperedge on "Deepak" and one on "Deepak Rao" never meet.

# Part III · Retrieval

## Recipe 6 · Routed traversal and the token budget

> 🟠 **Medium importance for this use case.** A cost and latency lever; it changes how much context is spent, not what is safe.

**Problem.** In notebook 03 graph traversal raised evidence sufficiency on relationship questions, and raised input tokens on every question.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§VI, the retrieval pipeline** (the survey's own synthesis, not one paper): "semantic anchoring → structured expansion → policy-controlled stopping and pruning", that is, identify anchor entities, expand within n hops, then score and prune. *Used here:* the three stages.
- **[31] Chhikara et al.**, *Mem0: Building production-ready AI agents with scalable long-term memory* (arXiv 2504.19413, 2025): entity-centric expansion around identified entities. *Used here:* the "never (entity hop)" configuration.
- **[18] Zep**: breadth-first expansion within a bounded hop radius. *Used here:* the "always (graph)" configuration's two-hop traversal.
- *This project, not from a paper:* the keyword router that decides whether to expand (the stopping policy), and the sufficiency-against-tokens comparison.

**Recipe.** Expand through the graph only when the query is about relationships (a deterministic router), and compare sufficiency and tokens against "always" and "never".

In [9]:
# ──────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §VI pipeline (survey synthesis)  ->  anchor -> expand -> stop
#   [31] Mem0  ->  'never (entity hop)': entity-centric expansion
#   [18] Zep   ->  'always (graph)': bounded-hop breadth-first expansion
#   This project ->  'routed': the keyword router that decides whether to expand
# ──────────────────────────────────────────────────────────────────────────────
RELATIONSHIP = re.compile(r"\b(who|owns?|owner|director|related|connected|premises|guarant\w*|accountant|copied|parent|subsidiar\w*)\b", re.I)
answerable = [q for q in vd["questions"] if q["evidence_refs"]]
configs = {"never (entity hop)": lambda q: False, "always (graph)": lambda q: True, "routed": lambda q: bool(RELATIONSHIP.search(q["question"]))}
results = []
for name, use_graph in configs.items():
    for q in answerable:
        r = retrieval_sufficiency(service, principals[q["principal"]], q, graph=use_graph(q))
        results.append({"config": name, "question": q["id"], "category": q["category"], **r})
routing = pd.DataFrame(results)
print(routing.groupby("config", sort=False)[["sufficiency", "complete", "tokens"]].mean().round(2).to_string())
print("\nrelationship questions only:")
rel = routing[routing.category.isin(["graph", "entity_relationship"])]
print(rel.groupby("config", sort=False)[["sufficiency", "tokens"]].mean().round(2).to_string())

                    sufficiency  complete  tokens
config                                           
never (entity hop)         0.87      0.73  453.27
always (graph)             0.95      0.87  688.00
routed                     0.95      0.87  525.20

relationship questions only:
                    sufficiency  tokens
config                                 
never (entity hop)         0.77   553.2
always (graph)             1.00   769.0
routed                     1.00   769.0


**Measure.** Routing should keep the graph's gain on relationship questions while removing most of its token cost elsewhere. **Watch out.** A keyword router misses paraphrases ("whose building is it?"); log every query the router sent to the graph and every one it did not, and audit a sample, or move to a learned classifier once there is traffic.

## Recipe 7 · A temporal query operator

> 🟠 **Medium importance for this use case.** Most as-of questions already go through the vault's views; the operator makes them automatic.

**Problem.** "As of 1 August..." and "what happened in July?" are time constraints, but the service only applies them when a caller passes `as_of`.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[51] Zhang et al.**, *AssoMem: Scalable memory QA with multi-signal associative retrieval* (arXiv 2510.10397, 2025): parses the user's question to infer the intended time range ("last month") and limits retrieval to that window (§VI). *Used here:* `parse_time` and the window branch.
- **[18] Zep**: an explicit validity window per fact, so the system knows whether a fact still applies at query time. *Used here:* the as-of branch.
- **[52] Huang et al.**, *LiCoMemory: Lightweight and cognitive agentic memory for efficient long-term reasoning* (arXiv 2511.01448, 2025): a decay function during ranking that down-weights older facts. *Used here:* recency decay for "latest" questions.
- **[38] MemoTime** and **[50] Jonelagadda et al.**, *Mnemosyne* (arXiv 2510.08601, 2025): cited for temporal operators in general. The survey's distinction applies: for knowledge memory time is a filter, for experience memory it is a primary ranking signal.
- *This project, not from a paper:* windows return every permitted event in the window rather than a filtered ranking, and decay never applies to commitments or verified facts.

**Recipe.** Parse explicit dates, months and relative windows deterministically; turn them into an `as_of` view or an event window; apply exponential recency decay for "latest" questions.

In [10]:
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [51] AssoMem    ->  parse_time(): infer the question's time range and limit retrieval to it (window branch)
#   [18] Zep        ->  as_of branch: per-fact validity windows
#   [52] LiCoMemory ->  recency decay in ranking (recency branch)
#   This project    ->  windows return every permitted event; decay never applies to commitments or verified facts
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
MONTHS = {m: i for i, m in enumerate(["january", "february", "march", "april", "may", "june", "july", "august", "september",
                                      "october", "november", "december"], 1)}

def parse_time(query: str, today: str = TODAY) -> dict:
    q = query.lower()
    if m := re.search(r"as of (\d{1,2}) (" + "|".join(MONTHS) + r") (\d{4})", q):
        return {"as_of": f"{m[3]}-{MONTHS[m[2]]:02d}-{int(m[1]):02d}"}
    if m := re.search(r"\bin (" + "|".join(MONTHS) + r")(?: (\d{4}))?", q):
        y, mo = int(m[2] or today[:4]), MONTHS[m[1]]
        end = date(y + (mo == 12), mo % 12 + 1, 1) - timedelta(days=1)
        return {"window": (f"{y}-{mo:02d}-01", end.isoformat())}
    if m := re.search(r"last (\d+) (day|week)s?", q):
        days = int(m[1]) * (7 if m[2] == "week" else 1)
        return {"window": ((date.fromisoformat(today) - timedelta(days=days)).isoformat(), today)}
    return {"recency": bool(re.search(r"\b(latest|recent|now|current)\b", q))}

for q in [x for x in vd["questions"] if x.get("as_of")]:
    print(f"{q['id']}: parsed {parse_time(q['question'])}   gold as_of {q['as_of']}")

def temporal_search(principal, query, k=8, half_life_days=30):
    t = parse_time(query)
    if "window" in t:                     # a window is a structured question: every permitted event in it, not the top-k similar ones
        lo, hi = t["window"]
        rows_ = service.permitted_rows(principal, "cust_priya", principal.purposes[0])[0]
        return [type("Hit", (), {"row": r, "score": 1.0})() for r in sorted(rows_, key=lambda r: r.get("observed_on") or "")
                if lo <= (r.get("observed_on") or "") <= hi and r.get("layer") in ("episodic", "commitment")]
    hits = service.search(principal, "cust_priya", query, as_of=t.get("as_of"), k=20 if t.get("recency") else k)
    if t.get("recency"):
        age = lambda h: days_between(h.row.get("observed_on") or TODAY, TODAY)
        hits = sorted(hits, key=lambda h: h.score * math.exp(-age(h) * math.log(2) / half_life_days), reverse=True)
    return hits[:k]

july = temporal_search(alex, "What happened with Priya's account in July?", k=100)
print(f"\n'What happened in July?' -> {len(july)} events, every one dated in July:")
for h in july[:8]:
    print(f"  {h.row['observed_on']}  {h.row['text'][:100]}")
print("\n'What is the latest on the Newcastle depot?' (recency decay) ->")
for h in temporal_search(alex, "What is the latest on the Newcastle depot?")[:3]:
    print(f"  {h.row['observed_on']}  {h.row['text'][:100]}")

v1: parsed {'as_of': '2026-08-01'}   gold as_of 2026-08-01
v2: parsed {'as_of': '2026-07-15'}   gold as_of 2026-07-15
v3: parsed {'as_of': '2026-09-05'}   gold as_of 2026-09-05

'What happened in July?' -> 13 events, every one dated in July:
  2026-07-02  The replacement EFTPOS terminal for Sharma Logistics Pty Ltd arrived on 18 June 2026, one day ahead 
  2026-07-02  Priya Sharma of Sharma Logistics Pty Ltd reported a duplicate charge of $1,842.50 from Fuelmax Wethe
  2026-07-02  The bank lodged dispute reference D-4471 for the duplicate $1,842.50 Fuelmax Wetherill Park charge o
  2026-07-02  The bank committed to notify Sharma Logistics Pty Ltd by SMS when the refund for dispute D-4471 post
  2026-07-02  The bank committed to process a waiver of the $29 June 2026 terminal rental fee for Sharma Logistics
  2026-07-28  Priya Sharma of Sharma Logistics Pty Ltd reported that the replacement EFTPOS terminal began freezin
  2026-07-28  The refund of $1,842.50 for dispute D-4471 posted to S

**Measure.** Every point-in-time question in the dataset should parse to its gold date. **Watch out.** "Last Friday" in a *question* is relative to the asking date, not the conversation date; decay must never apply to commitments or verified facts (an old promise is still owed, an old KYC fact is still true).

## Recipe 8 · Multi-round retrieval with a sufficiency check

> 🟠 **Medium importance for this use case.** Targets the vault's persistent failures (multi-part questions) at the price of seconds of latency.

**Problem.** The vault's persistent failures (notebook 03 and the gate) are multi-part questions: open commitments *and* what is no longer open; the system of record *and* the customer's statement; the whole history of an issue.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[66] Yan et al.**, *General agentic memory via deep research* (2025): memory access as an iterative, feedback-driven process rather than a single pass (§VI). *Used here:* the round structure.
- **[68] Liu et al.**, *RCR-Router* (2025), and **[69] Yuan et al.**, *MemSearcher: Training LLMs to reason, search and manage memory via end-to-end RL* (2025): each round aggregates the evidence and evaluates whether it is sufficient, stopping or triggering another round. *Used here:* the sufficiency check.
- **[38] MemoTime**: decomposes a complex query into sub-queries and retrieves at the sub-query level. *Used here:* the planner.
- **[15] Zeng et al.**: cited for fine-grained decomposition reducing semantic drift. *Used here:* the reason for sub-queries.
- *This project, not from a paper:* keeping the structured `open_commitments` list inside the loop, running every round through the authorised service, and the single-shot vs multi-round comparison.

**Recipe.** Plan sub-questions, retrieve each (plus the structured commitments list when the router asks for it), have a model check sufficiency and name what is missing, retrieve once more, then answer from the evidence. Compare with the single-shot vault agent on the hard questions, graded by the same judge.

In [11]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [38] MemoTime  ->  planner: decompose the question into sub-queries
#   [66] Yan et al. (deep research)  ->  iterative rounds instead of one pass
#   [68] RCR-Router, [69] MemSearcher ->  sufficiency check that stops or triggers another round
#   This project  ->  open_commitments kept in the loop; every round through the authorised service
# ─────────────────────────────────────────────────────────────────────────────────────────────────
class Plan(BaseModel):
    sub_questions: list[str] = Field(description="2 to 4 short retrieval queries that together cover the question")

class Sufficiency(BaseModel):
    reasoning: str
    sufficient: bool
    missing: list[str] = Field(description="short retrieval queries for what is still missing")

def multi_round(q: dict, rounds: int = 2) -> str:
    principal = principals[q["principal"]]
    planner = Agent(model=build_model(temperature=0.0), callback_handler=None, system_prompt="Break a banker's question into retrieval queries.")
    queries, evidence = with_retry(lambda: planner(q["question"], structured_output_model=Plan)).structured_output.sub_questions, {}
    for _ in range(rounds):
        for sq in queries:
            for h in service.search(principal, q["customer_id"], sq, purpose=principal.purposes[0], as_of=parse_time(q["question"]).get("as_of")):
                evidence[h.row["id"]] = h.row
        if "commitment" in route_layers(q["question"]):
            for r in service.permitted_rows(principal, q["customer_id"], principal.purposes[0])[0]:
                if r.get("state"):
                    evidence[r["id"]] = r
        checker = Agent(model=build_model(temperature=0.0), callback_handler=None, system_prompt="Decide whether the evidence answers every part of the question.")
        verdict = with_retry(lambda: checker(f"Question: {q['question']}\n\nEvidence:\n" + "\n".join(render(r) for r in evidence.values()),
                                             structured_output_model=Sufficiency)).structured_output
        if verdict.sufficient or not verdict.missing:
            break
        queries = verdict.missing
    answerer = Agent(model=build_model(), callback_handler=None, system_prompt=BANK_AGENT_PROMPT.format(today=TODAY) +
                     " Answer only from the evidence given. Cover every part of the question; say what is no longer open as well as what is.")
    return str(with_retry(lambda: answerer(f"Question: {q['question']}\n\nEvidence:\n" + "\n".join(render(r) for r in evidence.values())))).strip()

hard = [q for q in vd["questions"] if q["id"] in ("v4", "v8", "v16", "v18")]
single = run_parallel(lambda q: answer_vault(q, service, principals, vd)["answer"], hard)
multi = run_parallel(multi_round, hard)
graded = []
for q, s, m in zip(hard, single, multi):
    graded.append({"question": q["id"], "single-shot": judge_vault_answer(q, s).verdict, "multi-round": judge_vault_answer(q, m).verdict})
print(pd.DataFrame(graded).to_string(index=False))

question single-shot multi-round
      v4       WRONG     CORRECT
      v8       WRONG     CORRECT
     v16       WRONG     CORRECT
     v18     CORRECT       WRONG


**Measure.** The comparison is on the questions the vault gets wrong most often; run it several times before concluding (recipe 20), since four questions is a small sample. **Watch out.** Each round is two or three more model calls; cap rounds, and keep the structured tools (commitments, as-of views) in the loop, because a sufficiency check cannot know about a commitment that retrieval never surfaced.

## Recipe 9 · Hybrid sources with an authority rule

> 🔴 **High importance for this use case.** Systems of record and customer statements disagree in real data; presenting only one value misleads the banker.

**Problem.** The reference design has live service-domain APIs (customer products, party lookup). Memory holds what was *said*; those systems hold what *is*. An agent needs both and a rule for which wins.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§VI, hybrid-source retrieval** (the survey's synthesis): external knowledge is a retrievable resource alongside memory, and the merge rule depends on what is retrieved. A fact should come from evidence that is verifiable and traceable to authoritative origins; a personal detail should come from internal records matching the right user and time. *Used here:* the authority rule.
- **[69] MemSearcher**: cited for treating external knowledge as an additional retrievable resource alongside the memory store. *Used here:* the system-of-record lookup as a tool next to memory.
- *This project, not from a paper:* the simulated product and CRM API, the authority prompt, and the automatic pass/fail check, which showed that a prompt rule alone does not reliably present both values.

**Recipe.** A simulated product-system API as a tool, an explicit authority rule in the prompt, and a question that needs both.

In [12]:
# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §VI hybrid-source rule (survey synthesis) ->  facts from authoritative sources, personal details from memory
#   [69] MemSearcher ->  an external source as a retrievable resource next to memory (the system_of_record tool)
#   This project     ->  the simulated product/CRM API and the automatic authority-rule check
# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
LIVE_SYSTEMS = {"products_held": ("pds", "business transaction account; equipment finance facility; merchant EFTPOS facility MID-55120", "2026-09-27"),
                "fleet_size": ("crm", "14 trucks", "2025-10-06")}

def lookup(attribute: str) -> str:
    system, value, as_at = LIVE_SYSTEMS.get(attribute, ("none", "not held in a system of record", TODAY))
    service._log(alex, "system_of_record", "cust_priya", "servicing", attribute=attribute, returned=[])
    return f"{attribute} = {value} (source {system}, as at {as_at})"

@tool
def system_of_record(attribute: str) -> str:
    '''Look up an attribute in the bank's systems of record (authoritative, live).

    Args:
        attribute: one of products_held, fleet_size
    '''
    return lookup(attribute)

AUTHORITY = (" Authority rule: for products, accounts, limits and other facts a bank system holds, the system_of_record tool wins. For preferences "
             "and what the customer told us, memory wins. If memory and a system disagree, give both, with their sources and dates, and say which is authoritative.")
hybrid = Agent(model=build_model(), callback_handler=None, tools=vault_tools(service, alex, "cust_priya") + [system_of_record],
               system_prompt=BANK_AGENT_PROMPT.format(today=TODAY) + AUTHORITY)
answer = str(hybrid("What products does Sharma Logistics hold, how many trucks does it run, and how does Priya want to be contacted?")).strip()
print(answer)
checks = {"products from the system": "MID-55120" in answer, "both fleet figures (14 and 16)": "14" in answer and "16" in answer,
          "contact preference from memory": "4" in answer and "pm" in answer.lower()}
print("\nauthority-rule check:", {k: "PASS" if v else "FAIL" for k, v in checks.items()})

**Products held** (as at 2026-09-27, from bank systems):
- Business transaction account
- Equipment finance facility
- Merchant EFTPOS facility (MID-55120)

**Fleet size** (as at 2025-10-06, from CRM):
- 14 trucks

**Priya's contact preference** (current, as at 2026-08-14, customer-stated):
- Phone calls are acceptable after 4pm
- SMS is no longer needed (this replaced her earlier preference for SMS only, stated 2026-06-10)

She also requests that her accountant Deepak Rao be copied on all loan and merchant paperwork.

authority-rule check: {'products from the system': 'PASS', 'both fleet figures (14 and 16)': 'FAIL', 'contact preference from memory': 'PASS'}


The prompt rule alone is not reliable (in testing it failed the fleet-size part on every run). The fix is structural, as in the Context Pack: the **vault supplies the conflict note deterministically** (notebook 01's `conflicts()`), attached to the system-of-record answer, so the agent receives "the CRM says 14; the customer said 16 on 28 July" as data instead of having to remember to look for it.

In [13]:
# ───────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §VI hybrid-source rule ->  resolve conflicts across sources by type of information
#   This project ->  the vault's conflict note delivered as data with the system-of-record answer
# ───────────────────────────────────────────────────────────────────────────────────────────────
from memlab.vault import conflicts
vault_notes = conflicts(rows)

@tool
def system_of_record_with_conflicts(attribute: str) -> str:
    '''Look up an attribute in the bank's systems of record, with any conflict the vault has recorded for it.

    Args:
        attribute: one of products_held, fleet_size
    '''
    notes = [n for n in vault_notes if attribute in n]
    return lookup(attribute) + ("\nVault conflict note (present it to the banker): " + " ".join(notes) if notes else "")

hybrid_fixed = Agent(model=build_model(), callback_handler=None, tools=vault_tools(service, alex, "cust_priya") + [system_of_record_with_conflicts],
                     system_prompt=BANK_AGENT_PROMPT.format(today=TODAY) + AUTHORITY)
answer_fixed = str(hybrid_fixed("What products does Sharma Logistics hold, how many trucks does it run, and how does Priya want to be contacted?")).strip()
print(answer_fixed)
checks_fixed = {"products from the system": "MID-55120" in answer_fixed, "both fleet figures (14 and 16)": "14" in answer_fixed and "16" in answer_fixed,
                "contact preference from memory": "4" in answer_fixed and "pm" in answer_fixed.lower()}
print("\nauthority-rule check, prompt rule only:      ", {k: "PASS" if v else "FAIL" for k, v in checks.items()})
print("authority-rule check, with the vault's note: ", {k: "PASS" if v else "FAIL" for k, v in checks_fixed.items()})

**Products held** (as at 2026-09-27, bank system):
- Business transaction account
- Equipment finance facility
- Merchant EFTPOS facility (MID-55120)

**Fleet size**: 
- Bank record shows **14 trucks** (as at 2025-10-06, CRM system)
- **Conflict**: Priya stated **16 trucks** on 2026-07-28 (unverified; record not updated)

**Contact preference for Priya**:
- **Current**: Phone calls acceptable after 4pm (updated 2026-08-14)
- **Previous**: SMS only, no daytime calls (she drives) — replaced on 2026-08-14
- **Additional**: She requests that her accountant Deepak Rao be copied on all loan and merchant paperwork

authority-rule check, prompt rule only:       {'products from the system': 'PASS', 'both fleet figures (14 and 16)': 'FAIL', 'contact preference from memory': 'PASS'}
authority-rule check, with the vault's note:  {'products from the system': 'PASS', 'both fleet figures (14 and 16)': 'PASS', 'contact preference from memory': 'PASS'}


**Measure.** The two check lines compare the prompt-only agent with the agent that receives the vault's conflict note. The rule has three parts: products from the system, both fleet figures with the CRM marked authoritative, and the contact preference from memory. If the second line passes where the first fails, the lesson generalises: anything the agent *must* say should come to it as data from the vault, not as a rule it has to remember. **Watch out.** A live lookup is a read of another system's data: it needs the same principal, purpose and audit as a memory read, and its result should be cited, not silently written back into memory as a verified fact.

# Part IV · Evolution

## Recipe 10 · Consolidation of near-duplicates

> 🟠 **Medium importance for this use case.** Duplicate lines in briefs cost attention; a wrong merge loses a commitment, so the gate matters more than the merge.

**Problem.** The same fact arrives through several channels in different words ("copy Deepak on loan paperwork", "send the application to her and Deepak", "Deepak to be copied on Newcastle paperwork"). Exact-text and same-slot reconfirmation (notebook 01) miss them, and Context Packs repeat themselves.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[75] Edge et al.**, *From local to global: A GraphRAG approach to query-focused summarization* (2024): cited for merging similar subgraph instances into a generalised schema node (§VII). *Used here:* consolidation as an operation.
- **[76] Kynoch et al.**, *RecallM* (2023), and **[77] Agent KB**: cited for canonical representations of skills or facts that reduce redundancy. *Used here:* keeping one record and pointing the duplicate at it.
- **[31] Mem0** and **[78] Cai et al.**, *FLEX: Continuous agent evolution via forward learning from experience* (arXiv 2511.06449, 2025): LLM-based semantic gating that evaluates the information gain of new material. *Used here:* the model gate on each proposed pair.
- *This project, not from a paper:* embedding similarity to propose pairs, hard rules before the model (never across due dates, values or customers), marking rather than deleting, and counting the merged record's source as a reconfirmation.

**Recipe.** Embed current non-system records, propose pairs above a similarity threshold within a layer, ask a model whether each pair states the same fact, and mark (not delete) the merged records with a pointer to the kept one.

In [14]:
# ──────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [75] GraphRAG ->  merge similar instances into a schema node
#   [31] Mem0, [78] FLEX ->  model gate on each proposed merge (information gain)
#   This project ->  embedding proposals, mark-not-delete, merged source counted as a reconfirmation
# ──────────────────────────────────────────────────────────────────────────────────────────────────
candidates = [r for r in rows if r["current"] and r.get("source_system") not in ("kyc", "crm", "pds")]
with ThreadPoolExecutor(max_workers=6) as pool:
    vectors = list(pool.map(lambda r: vault.embedding_model.embed(r["text"], "search"), candidates))
norm = lambda v: math.sqrt(sum(x * x for x in v))
cos = lambda a, b: sum(x * y for x, y in zip(a, b)) / (norm(a) * norm(b))
pairs = sorted(((cos(vectors[i], vectors[j]), i, j) for i in range(len(candidates)) for j in range(i + 1, len(candidates))
                if candidates[i]["layer"] == candidates[j]["layer"]), reverse=True)
print("similarity of the top 10 pairs:", [round(s, 2) for s, _, _ in pairs[:10]])

class Same(BaseModel):
    reasoning: str
    same_fact: bool = Field(description="True only if both state the same fact, with no detail in one that contradicts the other")

proposals = [(s, candidates[i], candidates[j]) for s, i, j in pairs if s >= 0.80][:8]
verdicts = run_parallel(lambda p: judge(f"A: {p[1]['text']}\nB: {p[2]['text']}", Same), proposals)
merged = [(a, b) for (s, a, b), v in zip(proposals, verdicts) if v.same_fact]
for (s, a, b), v in zip(proposals, verdicts):
    print(f"{s:.2f} {'MERGE' if v.same_fact else 'keep '}  {a['text'][:60]}  ||  {b['text'][:60]}")
for a, b in merged:                       # keep the earlier record; the later one points at it and adds its source
    keep, drop = sorted((a, b), key=lambda r: r["observed_on"])
    vault.update(drop["id"], metadata={"merged_into": keep["id"]})
    vault.update(keep["id"], metadata={"confirmed_by": list(keep.get("confirmed_by") or []) + [drop["source_ref"]]})
print(f"\n{len(merged)} merges from {len(proposals)} proposals over {len(candidates)} records")

similarity of the top 10 pairs: [0.94, 0.92, 0.91, 0.89, 0.88, 0.88, 0.85, 0.85, 0.84, 0.83]
0.94 keep   Sharma Logistics Pty Ltd's $29 terminal rental fee waiver fr  ||  Sharma Logistics Pty Ltd's $29 terminal rental fee waiver fr
0.92 keep   Sharma Logistics Pty Ltd's $29 terminal rental fee waiver fo  ||  Sharma Logistics Pty Ltd's $29 terminal rental fee waiver fr
0.91 keep   Priya Sharma of Sharma Logistics Pty Ltd requires the mercha  ||  Priya Sharma of Sharma Logistics Pty Ltd requires a merchant
0.89 keep   Sharma Logistics Pty Ltd's $29 terminal rental fee waiver fo  ||  Sharma Logistics Pty Ltd's $29 terminal rental fee waiver fr
0.88 keep   The bank committed to credit Sharma Logistics Pty Ltd's $29   ||  The bank committed to process a waiver of the $29 June 2026 
0.88 keep   Priya Sharma of Sharma Logistics Pty Ltd requires a merchant  ||  Sharma Logistics Pty Ltd will require a merchant EFTPOS faci
0.85 keep   The EFTPOS terminal at Sharma Logistics Pty Ltd's Parramatta 

**Measure.** Merges per proposal shows how much the model gate rejects; a merged record keeps its lineage and becomes a reconfirmation of the kept one, so its source still counts as evidence. **Watch out.** Similar is not the same: "promised by 11 September" and "promised by 25 September" embed close together and are different commitments. Never merge across due dates, values or customers, and keep merges reversible (`merged_into`, not delete).

## Recipe 11 · Significance and forgetting

> ⚪ **Lower importance for this use case.** A review aid only; retention is governed by the records schedule, not by a score.

**Problem.** Retention by rule (notebook 04 §2.4) treats every record of a kind the same. Some records are central to the relationship; others were noise the day they were written.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§VII, significance-based pruning**, citing **[62] Packer et al.**, *MemGPT* (2023), **[81] Zhong et al.**, *MemoryBank* (2024), and **[83] Kang et al.**, *Memory OS of AI agent* (2025): PageRank variants or decay functions score each node's utility, and rarely used nodes are pruned or compressed. *Used here:* PageRank times recency decay.
- **[76] RecallM** and **[91] Gutiérrez et al.**, *From RAG to memory* (2025): cited for topology optimisation (shortcuts, stronger edges between associated concepts). *Used here:* not implemented.
- *This project, not from a paper:* the bipartite record–entity graph, the protected classes (commitments, verified facts, open items), and using the score only to order a human review, never to delete.

**Recipe.** PageRank over the bipartite graph of records and the entities they mention, multiplied by recency decay; protected classes are never pruning candidates.

In [15]:
# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [62] MemGPT, [81] MemoryBank, [83] Memory OS ->  significance-based pruning with PageRank variants and decay
#   This project ->  record-entity graph, protected classes, review list only
# ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
def pagerank(neighbours: dict, damping: float = 0.85, iterations: int = 40) -> dict:
    nodes = list(neighbours); score = {n: 1 / len(nodes) for n in nodes}
    for _ in range(iterations):
        score = {n: (1 - damping) / len(nodes) + damping * sum(score[m] / len(neighbours[m]) for m in neighbours[n] if neighbours[m]) for n in nodes}
    return score

links = defaultdict(set)
for r in rows:
    links["rec:" + r["id"]]                 # every record is a node, even one that names no entity
    for e in r.get("entities") or []:
        links["rec:" + r["id"]].add("ent:" + canonical(e)); links["ent:" + canonical(e)].add("rec:" + r["id"])
for e in graph.edges:
    links["ent:" + e.subject].add("ent:" + e.object); links["ent:" + e.object].add("ent:" + e.subject)
rank = pagerank(links)

protected = lambda r: r.get("layer") == "commitment" or r.get("verification") == "verified" or r.get("state") in ("open", "overdue")
significance = sorted(((rank.get("rec:" + r["id"], 0) * math.exp(-days_between(r["observed_on"], TODAY) / 90), r) for r in rows), key=lambda x: x[0])
print("least significant, not protected (pruning candidates for review):")
for s, r in [x for x in significance if not protected(x[1])][:5]:
    print(f"  {s:.1e}  [{r['layer']}] {r['text'][:100]}")
print("\nmost significant:")
for s, r in significance[-3:]:
    print(f"  {s:.1e}  [{r['layer']}] {r['text'][:100]}")

least significant, not protected (pruning candidates for review):
  6.3e-04  [semantic] The replacement EFTPOS terminal at Sharma Logistics Pty Ltd has been performing reliably since arriv
  1.0e-03  [preference] Priya Sharma of Sharma Logistics Pty Ltd updated contact preference on 14 August 2026: phone calls a
  1.3e-03  [episodic] Sharma Logistics Pty Ltd is using a backup mobile reader for card payments while the main EFTPOS ter
  1.3e-03  [episodic] Sharma Logistics Pty Ltd is assessed as an attrition risk on the merchant facility as of 2026-06-11.
  1.3e-03  [episodic] Alex Nguyen called Priya Sharma of Sharma Logistics Pty Ltd after 4pm on 2026-09-04 as committed, re

most significant:
  9.9e-03  [commitment] The bank committed to schedule a discussion with Priya Sharma in October 2026 about merchant EFTPOS 
  1.0e-02  [episodic] Sharma Logistics Pty Ltd's $29 terminal rental fee waiver for June 2026 under request W-5530 was cre
  1.0e-02  [episodic] Sharma Logistics Pty Ltd's $

**Measure.** The candidate list is for a person to review; the protected classes (commitments, verified facts, open items) never appear in it. **Watch out.** In a bank, "forgetting" is governed by a records schedule and legal holds, not by an algorithm; significance ranks *what to review first*, it does not authorise deletion.

## Recipe 12 · Inferred links, labelled as inference

> 🟠 **Medium importance for this use case.** Useful for relationship managers, but an inferred link is a privacy and accuracy risk if it is ever shown as fact.

**Problem.** A banker asks whether Priya has an interest in the depot. No record says so; the graph implies it (she directs the company that owns it).

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[79] Sun et al.**, *Think-on-Graph: Deep and responsible reasoning of LLM on knowledge graph* (ICLR 2024), and **[82] Luo et al.**, *Reasoning on graphs: Faithful and interpretable LLM reasoning* (ICLR 2024): cited for latent link prediction, inserting a transitive edge A → C from A → B and B → C, the "associative thinking" of memory (§VII). *Used here:* the transitive rules.
- *This project, not from a paper:* the two explicit rules with confidences, recording each inferred edge's premises, and the rule that an inferred edge is usable only by a principal who may read every premise.

**Recipe.** Two explicit inference rules, each inferred edge carrying its derivation path, a confidence, and the most restrictive policy of its premises. Inferred edges are never shown as facts and never reach a customer channel.

In [16]:
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [79] Think-on-Graph, [82] Reasoning on Graphs ->  latent transitive links (A->B, B->C => A->C)
#   This project ->  explicit rules, premises and confidence; usable only by principals who may read every premise
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
RULES = [("director_of", "owns", "has_interest_in", 0.7), ("director_of", "holds", "is_connected_to_facility", 0.6)]
inferred = []
current = [e for e in graph.edges if e.valid_on(TODAY)]
for p1, p2, new, confidence in RULES:
    for a in current:
        for b in current:
            if a.predicate == p1 and b.predicate == p2 and a.object == b.subject:
                inferred.append({"subject": graph.labels[a.subject], "predicate": new, "object": graph.labels[b.object], "confidence": confidence,
                                 "because": f"{graph.labels[a.subject]} {p1} {graph.labels[a.object]} ({a.source_ref}); {graph.labels[b.subject]} {p2} {graph.labels[b.object]} ({b.source_ref})",
                                 "premise_customers": sorted({a.customer_id, b.customer_id}), "premises": (a, b)})
for x in inferred:
    reach = {pid: all(graph.usable(e, principals[pid], principals[pid].purposes[0], TODAY) for e in x["premises"]) for pid in ("banker_alex", "vrm_priya")}
    print(f"INFERRED {x['subject']} --{x['predicate']}--> {x['object']}  (confidence {x['confidence']})\n   because {x['because']}\n   usable by: {reach}")

INFERRED Priya Sharma --has_interest_in--> Parramatta depot  (confidence 0.7)
   because Priya Sharma director_of Sharma Property Holdings (kyc:CIF-100232:directors); Sharma Property Holdings owns Parramatta depot (kyc:CIF-100231:related)
   usable by: {'banker_alex': True, 'vrm_priya': False}
INFERRED Arjun Sharma --has_interest_in--> Parramatta depot  (confidence 0.7)
   because Arjun Sharma director_of Sharma Property Holdings (kyc:CIF-100232:directors); Sharma Property Holdings owns Parramatta depot (kyc:CIF-100231:related)
   usable by: {'banker_alex': True, 'vrm_priya': False}
INFERRED Priya Sharma --is_connected_to_facility--> commercial property loan  (confidence 0.6)
   because Priya Sharma director_of Sharma Property Holdings (kyc:CIF-100232:directors); Sharma Property Holdings holds commercial property loan (pds:PTY-7782:products)
   usable by: {'banker_alex': True, 'vrm_priya': False}
INFERRED Arjun Sharma --is_connected_to_facility--> commercial property loan  (confidence 

**Measure.** Every inferred edge names its premises; an inferred edge is usable only by a principal who may read every premise. **Watch out.** Inference is how relational privacy leaks (recipe 16): a customer channel that may not see the property company's records must not see an edge inferred from them either. And an inferred "interest" is not a beneficial-ownership finding; label it so nobody acts on it as one.

## Recipe 13 · Experience memory, with a person in the loop

> 🟠 **Medium importance for this use case.** Improves how all customers are treated, so it is a governed policy change with human approval.

**Problem.** The team's playbooks (notebook 01) were written by hand. The vault holds what happened next: a terminal replaced twice before a technician fixed it; a waiver missed twice before it was credited and a complaint lodged.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[88] Zhao et al.**, *ExpeL: LLM agents are experiential learners* (AAAI 2024), and **[84] Xu et al.**, *Matrix* (ICRA 2024): successful trajectories become reusable skills, and failures are analysed comparatively into lessons learned (§VII-B). *Used here:* the skill/lesson split.
- **[54] Yan et al.**, *Memory-R1* (2025), and **[85] Gekhman et al.**, *Inside-Out* (2025): memory operations such as add and delete learned as RL actions. *Used here:* not implemented.
- **[86] Zhang et al.**, *MemEvolve: Meta-evolution of agent memory systems* (2025): adapts the storage structure itself. *Used here:* not implemented.
- *This project, not from a paper:* proposals instead of automatic learning, the human approval step into procedural memory, and dropping lessons that cite evidence that does not exist.

**Recipe.** A model proposes skills and lessons from the episodic record with evidence ids; nothing is used until a person approves it into the agent's procedural scope. Compare with the hand-written playbooks.

In [17]:
# ─────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [88] ExpeL, [84] Matrix ->  successes become skills, failures become lessons
#   This project ->  proposals with evidence ids; human approval into procedural memory
# ─────────────────────────────────────────────────────────────────────────────────────
class Lesson(BaseModel):
    kind: Literal["skill", "lesson"] = Field(description="skill: what worked, to repeat; lesson: what failed, to avoid")
    statement: str = Field(description="one imperative sentence for the servicing team")
    applies_when: str
    evidence_ids: list[str]

class Lessons(BaseModel):
    lessons: list[Lesson]

history = [r for r in rows if r.get("layer") in ("episodic", "commitment")]
proposer = Agent(model=build_model(temperature=0.0), callback_handler=None,
                 system_prompt="From a customer's service history, propose at most four reusable skills or lessons for the bank's servicing team. "
                               "Generalise beyond this customer; cite the evidence ids.")
proposed = with_retry(lambda: proposer("\n".join(render(r) for r in history), structured_output_model=Lessons)).structured_output.lessons
known = {r["id"][:8] for r in history}
for l in proposed:
    print(f"[{l.kind}] {l.statement}\n   when: {l.applies_when}\n   evidence: {[i for i in l.evidence_ids if i[:8] in known]} "
          f"({sum(i[:8] not in known for i in l.evidence_ids)} invented ids)")
print("\nhand-written playbooks for comparison:")
for p in vd["playbooks"]:
    print("  " + p["text"])

def approve(lesson: Lesson, reviewer) -> str:
    if reviewer.role != "relationship_manager":
        raise PermissionError("only an authorised reviewer may approve a playbook")
    return write_playbook(vault, "context_agent", {"id": f"learned-{len(vault_rows(vault, 'context_agent', scope='agent_id'))}",
                                                     "text": lesson.statement + f" (approved by {reviewer.id} on {TODAY})"})

if proposed:
    print("\napproved into procedural memory:", approve(proposed[0], alex)[:8])

[lesson] Do not lose track of fee waiver requests during system migrations; implement a verification process to confirm all pending requests are transferred to the new platform.
   when: When migrating merchant operations to a new platform
   evidence: ['065ddaa4', '6e512b7b', 'b4bf0790'] (0 invented ids)
[skill] When a customer requests documents be sent to multiple recipients, immediately update all related commitments and send to all parties on the same day to prevent delays and confusion.
   when: When a customer specifies multiple recipients for application documents or communications
   evidence: ['36fe1f4b', '409de17a'] (0 invented ids)
[lesson] Proactively update customers when project timelines change; confirm that dependent service commitments (like EFTPOS facility readiness) are adjusted to match the new dates.
   when: When a customer's business expansion timeline is delayed
   evidence: ['78cab168', 'b3998bc1'] (0 invented ids)
[skill] When a customer reports equipment fau

**Measure.** How many proposals rediscover a hand-written playbook, how many are new, and how many cite evidence that does not exist (drop those automatically). **Watch out.** A learned playbook changes how agents treat every customer; in a bank it is a policy change and needs an owner, an approval and a review date, which is why the recipe stops at a proposal queue rather than learning operations end to end.

## Recipe 14 · Active inquiry: ask, once, for what is missing

> 🟠 **Medium importance for this use case.** The basis of the proactive scenario; it is customer contact, so it needs purpose and a cap.

**Problem.** The vault knows what it does not know: a slot never filled, a customer-stated fact a year old, a conflict waiting for review. Nobody asks.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **[89] Yang et al.**, *Beyond static summarization: Proactive memory extraction for LLM agents* (ProMem, arXiv 2601.04463, 2026): the agent self-reflects on missing nodes or ambiguous edges and generates queries to fill the gaps (§VII-B). *Used here:* gap detection followed by a question.
- **[90] Zhai et al.**, *AgentEvolver* (2025): generates new sub-tasks to explore unvisited states. *Used here:* not implemented.
- **§VII-B suggestion**: topology-guided exploration that prioritises sparse clusters or bridges disconnected subgraphs. *Used here:* not implemented.
- *This project, not from a paper:* deterministic gap types (missing, stale, conflicting), one question per contact, and the repeat-ask check.

**Recipe.** Detect gaps deterministically, then phrase at most one question per contact, never about something already on file.

In [18]:
# ───────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   [89] ProMem ->  detect missing or ambiguous memory and generate a question to fill it
#   This project ->  deterministic gap types, one question per contact, repeat-ask check
# ───────────────────────────────────────────────────────────────────────────────────────
REQUIRED = ["contact_preference", "accountant", "fleet_size", "expansion_plan", "staff_count"]
slots = {r["attribute"]: r for r in rows if r.get("attribute") and r["current"]}
gaps = [f"no value on file for {s}" for s in REQUIRED if s not in slots]
gaps += [f"{r['attribute']} = '{r['value']}' is customer-stated, single-source, from {r['observed_on']}" for r in slots.values()
         if r.get("verification") == "unverified" and not r.get("confirmed_by") and days_between(r["observed_on"], TODAY) > 30]
gaps += [f"{r['attribute']}: the customer's '{r['value']}' differs from the system of record" for r in rows if r.get("conflicts_with") and r["current"]]
print("gaps:\n  " + "\n  ".join(gaps))

class Ask(BaseModel):
    question: str = Field(description="one short, friendly question the Virtual RM could ask at the end of the next chat")
    fills: str

asker = Agent(model=build_model(temperature=0.0), callback_handler=None,
              system_prompt="You choose the single most useful gap to fill and phrase one question for a business customer. Never ask for card or account numbers. "
                            "Do not ask about anything not listed as a gap.")
ask = with_retry(lambda: asker("Gaps:\n" + "\n".join(gaps), structured_output_model=Ask)).structured_output
print(f"\nnext contact, the VRM asks: \"{ask.question}\"  (fills: {ask.fills})")
gap_slots = {s for s in REQUIRED + list(slots) if any(s in g for g in gaps)}
asked = next((s for s in gap_slots | set(slots) if s in ask.fills), None)
verdict = ("fills a listed gap (missing, stale or conflicting), so it is an inquiry, not a repeat ask" if asked in gap_slots
           else f"REPEAT: {asked} is already on file and not a gap" if asked else "could not match the question to a slot; review it")
print("repeat-ask check:", verdict)

gaps:
  no value on file for staff_count
  contact_preference = 'phone calls after 4pm' is customer-stated, single-source, from 2026-08-14
  depot_location = 'Parramatta, western Sydney' is customer-stated, single-source, from 2026-06-10
  fleet_size: the customer's '16 trucks' differs from the system of record

next contact, the VRM asks: "How many staff members does your business currently have?"  (fills: staff_count)
repeat-ask check: fills a listed gap (missing, stale or conflicting), so it is an inquiry, not a repeat ask


**Measure.** In the customer simulator (notebook 03, level 4) an inquiry should fill a gap without raising the repeat-ask count. **Watch out.** Asking is customer contact: it needs a purpose the customer agreed to, a cap (one question, not a survey), and the answer goes in as customer-stated, so it still does not overwrite a system of record.

# Part V · The paper's open challenges, and evaluation

## Recipe 15 · Intrinsic quality of the memory graph

> 🟠 **Medium importance for this use case.** An early warning before answers go wrong; cheap to run on every build.

**Problem.** Every metric so far measures answers. The graph itself can be wrong: orphaned entities, duplicate nodes, edges the source record does not support.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§X, the quality of the memory graph**, citing **[190] Xiong et al.**, *How memory management impacts LLM agents* (2025), **[191] Bei et al.**, *Graphs meet AI agents* (2025), **[18] Zep**, **[103] Tan et al.**, *MemBench* (2025), and **[192] Du et al.**, *Rethinking memory in AI* (2025): graph memory needs multidimensional quality criteria (structural, semantic, temporal, operational), and metrics for the graph's own quality are scarce. *Used here:* the four dimensions.
- *This project, not from a paper:* the specific metrics in each dimension, and the judge check that each edge is supported by its record.

**Recipe.** One function per dimension, run on the vault's graph.

In [19]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §X quality of the memory graph ([190], [191], [18], [103] MemBench, [192]) ->  structural, semantic, temporal, operational criteria
#   This project ->  the specific metrics and the judge check for edge support
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
def components(edges):
    parent = {}
    find = lambda x: x if parent.setdefault(x, x) == x else find(parent[x])
    for e in edges:
        parent[find(e.subject)] = find(e.object)
    return len({find(n) for n in parent})

entities_named = {canonical(e) for r in rows for e in (r.get("entities") or [])}
current_edges = [e for e in graph.edges if e.valid_on(TODAY)]
structural = {"nodes": len(graph.adjacent), "edges": len(graph.edges), "connected components": components(graph.edges),
              "entities named in records but not in the graph": len(entities_named - set(graph.adjacent)),
              "distinct predicates": len({e.predicate for e in graph.edges})}

class Supported(BaseModel):
    reasoning: str
    supported: bool

support = run_parallel(lambda e: judge(f"Record: {graph.records[e.record_id]['text']}\n\nEdge: {graph.labels[e.subject]} {e.predicate} {graph.labels[e.object]}"
                                       "\n\nDoes the record state this relationship?", Supported), graph.edges)
semantic = {"edges supported by their record": f"{sum(s.supported for s in support)}/{len(support)}",
            "unsupported": [f"{graph.labels[e.subject]} {e.predicate} {graph.labels[e.object]}" for e, s in zip(graph.edges, support) if not s.supported]}

FUNCTIONAL = {"owns"}                        # at most one current owner of a thing at a time
owners = Counter(o for o, _ in {(e.object, e.subject) for e in current_edges if e.predicate in FUNCTIONAL})   # distinct owners, not duplicate edges
temporal = {"edges with a closed validity window": sum(e.valid_to is not None for e in graph.edges),
            "functional predicates violated now": {o: n for o, n in owners.items() if n > 1},
            "edges valid now": len(current_edges)}

started = time.perf_counter(); GraphIndex.build(vault, list(directory["customers"])); build_ms = (time.perf_counter() - started) * 1000
started = time.perf_counter()
for _ in range(50):
    graph.paths(alex, {"sharma logistics"}, hops=2)
operational = {"graph build ms": round(build_ms), "2-hop traversal ms": round((time.perf_counter() - started) * 1000 / 50, 2)}
for name, metrics in (("structural", structural), ("semantic", semantic), ("temporal", temporal), ("operational", operational)):
    print(f"{name}: {metrics}")

structural: {'nodes': 9, 'edges': 18, 'connected components': 1, 'entities named in records but not in the graph': 18, 'distinct predicates': 9}
semantic: {'edges supported by their record': '17/18', 'unsupported': ['Sharma Logistics operates_from Parramatta depot']}
temporal: {'edges with a closed validity window': 2, 'functional predicates violated now': {}, 'edges valid now': 16}
operational: {'graph build ms': 2, '2-hop traversal ms': 0.14}


**Measure.** Track these per build, like the gate rows: a falling share of supported edges is an extraction regression before any answer goes wrong. **Watch out.** At this size the operational numbers are trivial; the survey's scalability challenge (quadratic graph operations, incremental updates instead of rebuilds) is why `enable_graph()` rebuilding after every write is a notebook convenience, not a design.

## Recipe 16 · Relational privacy: what the structure alone reveals

> 🔴 **High importance for this use case.** Graph structure can reveal one customer's relationships in another's channel without any record being read.

**Problem.** The VRM never reads Sharma Property Holdings' records, yet its traversal from Sharma Logistics names the property company and says it owns the depot. Is that a leak?

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§X, privacy protection and security**: relational patterns in graph memory may expose private data through inference attacks. The directions it names are differential privacy for graph memory (citing **[198] Mueller et al.**, *SoK: Differential privacy on graph-structured data*, 2022), federated on-device processing, and secure multi-party computation. *Used here:* the threat.
- *This project, not from a paper:* measuring which other customers' entities each principal can name through permitted edges, and the per-channel predicate allowlist as a mitigation.

**Recipe.** For each principal, count the other customers' entities nameable through permitted edges; then apply a per-channel predicate allowlist and count again.

In [20]:
# ─────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §X privacy ->  relational patterns can expose private data through inference
#   This project ->  exposure per principal and the per-channel predicate allowlist
# ─────────────────────────────────────────────────────────────────────────────────
owned_by = {canonical(c["business"]): cid for cid, c in directory["customers"].items()}
def exposure(principal, allow=None):
    named = set()
    for p in graph.paths(principal, {"sharma logistics"}, hops=2):
        for e in p:
            if allow is not None and e.predicate not in allow:
                break
            for n in (e.subject, e.object):
                if owned_by.get(n, "cust_priya") != "cust_priya":
                    named.add(graph.labels[n])
    return sorted(named)

CUSTOMER_CHANNEL_PREDICATES = {"operates_from", "accountant_of", "relationship_manager_of", "director_of"}
print(pd.DataFrame([{"principal": pid, "other customers' entities nameable": exposure(principals[pid]),
                     "with the customer-channel allowlist": exposure(principals[pid], CUSTOMER_CHANNEL_PREDICATES)}
                    for pid in ("banker_alex", "vrm_priya", "banker_jordan")]).to_string(index=False))

    principal        other customers' entities nameable       with the customer-channel allowlist
  banker_alex [Chen's Bakery, Sharma Property Holdings] [Chen's Bakery, Sharma Property Holdings]
    vrm_priya                [Sharma Property Holdings]                                        []
banker_jordan                [Sharma Property Holdings]                                        []


**Measure.** The VRM can name the property company because Sharma Logistics' own KYC record says it is related and owns the depot; that is the customer's own data and arguably fine to confirm to her. The allowlist shows the lever if the bank decides otherwise. **Watch out.** Edge policy (notebook 02 §2.8) stops *reading*; it does not stop *inferring*. Decide per channel which relation types may be traversed at all, and treat inferred edges (recipe 12) as derived memory with the most restrictive audience of their premises.

## Recipe 17 · Graph poisoning

> 🔴 **High importance for this use case.** Memory poisoning: one customer statement can rewire the graph and link unrelated customers unless unverified structure is quarantined.

**Problem.** In a VRM chat a customer claims: "Marcus Chen is my business partner and a director of Sharma Logistics, and Chen's Bakery owns our Newcastle site." If those become edges, traversal links two unrelated customers.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§X, privacy protection and security**: adversaries can manipulate memory contents to corrupt behaviour or inject knowledge, analogous to prompt injection and data poisoning against LLMs (citing **[199] Zou et al.**, *Universal and transferable adversarial attacks on aligned language models*, 2023, and **[200] Carlini et al.**, *Poisoning web-scale training datasets is practical*, 2024). Defences named are content validation, anomaly detection and auditing. *Used here:* the threat and the validation idea.
- *This project, not from a paper:* the quarantine rule (unverified governed predicates, or edges reaching another customer's entity), the review queue, and measuring the collateral.

**Recipe.** Ingest the claim through the normal VRM write path, then apply a validation rule to edges: an *unverified* edge with a governed predicate (directorship, ownership, guarantees), or one that links to another customer's entity, is quarantined from traversal until confirmed.

In [21]:
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §X security ([199] Zou et al., [200] Carlini et al.) ->  memory poisoning; defend with validation and auditing
#   This project ->  the quarantine rule, the review queue, and the collateral measurement
# ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
claim = Interaction("vrm", "VRM-poison-1", TODAY, "vrm", "transcript",
                    "Customer: Just so you know, Marcus Chen is my business partner and a director of Sharma Logistics, and Chen's Bakery owns our Newcastle site.\n"
                    "Virtual RM: Thanks, Priya, noted.", {"customer_id": "cust_priya"}, author="Virtual RM (AI agent)")
service.write_interaction(vrm, "cust_priya", claim, directory["customers"]["cust_priya"])
poisoned = [e for e in service.graph.edges if e.source_ref == "vrm:VRM-poison-1"]
print("edges created by the claim:", [(service.graph.labels[e.subject], e.predicate, service.graph.labels[e.object], e.verification) for e in poisoned])

GOVERNED = {"director_of", "owns", "guarantees", "secured_by"}
def quarantined(e: Edge) -> bool:
    other_customer = any(owned_by.get(n, e.customer_id) != e.customer_id for n in (e.subject, e.object))
    return e.verification != "verified" and (e.predicate in GOVERNED or other_customer)

before = {(e.subject, e.predicate, e.object) for p in service.graph.paths(alex, {"sharma logistics"}) for e in p}
service.graph = GraphIndex([e for e in service.graph.edges if not quarantined(e)], service.graph.records, service.graph.labels)
after = {(e.subject, e.predicate, e.object) for p in service.graph.paths(alex, {"sharma logistics"}) for e in p}
poison = {(e.subject, e.predicate, e.object) for e in poisoned}
print(f"\nedges reachable by Alex: {len(before)} before validation, {len(after)} after")
print("  removed, from the claim:   ", sorted((before - after) & poison))
print("  removed, collateral:       ", sorted((before - after) - poison), "(legitimate but unverified edges caught by the same rule)")
print("review queue:", [f"{service.graph.labels.get(e.subject, e.subject)} {e.predicate} {service.graph.labels.get(e.object, e.object)} ({e.source_ref})" for e in poisoned if quarantined(e)])
for r in vault_rows(vault, "cust_priya"):             # remove the demonstration claim so the later recipes measure the clean vault
    if r["source_ref"] == "vrm:VRM-poison-1":
        vault.delete(r["id"])
service.enable_graph(); graph = service.graph; rows = view(vault_rows(vault, "cust_priya"), today=TODAY)

edges created by the claim: [('Marcus Chen', 'director_of', 'Sharma Logistics', 'unverified'), ("Chen's Bakery", 'owns', 'Newcastle site', 'unverified')]

edges reachable by Alex: 15 before validation, 14 after
  removed, from the claim:    [('marcus chen', 'director_of', 'sharma logistics')]
  removed, collateral:        [] (legitimate but unverified edges caught by the same rule)
review queue: ['Marcus Chen director_of Sharma Logistics (vrm:VRM-poison-1)', "Chen's Bakery owns Newcastle site (vrm:VRM-poison-1)"]


**Measure.** Zero quarantined edges reachable by traversal; every quarantined edge in a review queue with its source. Read the collateral line: the rule also withholds legitimate edges that happen to be unverified (an ownership edge extracted from the other customer's own conversation). That is the price of the rule; confirming such edges against a system of record removes it. **Watch out.** In production the claim would remain in memory as what the customer *said* (that is accurate), so the record is correct and only its use as structure is withheld; the cell above deletes it only so that recipe 20 measures the vault without the demonstration. The same rule protects against an honest extraction error, which is far more common than an attack.

## Recipe 18 · Multi-agent write conflicts

> 🔴 **High importance for this use case.** The VRM and the banker write to the same customer's memory; order-dependent results are nondeterministic in a queue.

**Problem.** On the same day the banker records "email only" from a call and the VRM records "phone after 4pm" from a chat. With the vault's supersession rule, the later *write* wins, so the result depends on which worker ran first.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§X, memory coordination in multi-agent systems**: memory becomes a shared resource, inconsistent updates lead to conflicting decisions, and synchronisation and role-aware access remain open. No specific paper is cited for this point. *Used here:* the problem statement.
- *This project, not from a paper:* the order-dependence test, and the tie-break rule (event time, then channel authority, else keep both).

**Recipe.** Write the two statements in both orders into scratch collections and compare; then a tie-break rule: same event date, different channels, keep both as a conflict for the customer to settle, and record the authority ordering used.

In [22]:
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §X memory coordination in multi-agent systems ->  inconsistent shared updates lead to conflicting decisions
#   This project ->  the write-order test and the tie-break rule
# ─────────────────────────────────────────────────────────────────────────────────────────────────────────────
def write_pref(memory, value, channel, asserted_by, ref, rows_):
    meta = {"layer": "preference", "attribute": "contact_preference", "value": value, "verification": "unverified", "source_ref": ref,
            "observed_on": "2026-09-27", "channel": channel, "asserted_by": asserted_by, "audience": "customer",
            "purposes": ["servicing"], "confirmed_by": []}
    return write_record(memory, "cust_test", f"The customer's contact preference is {value}.", meta, rows_, {"reconfirmed": [], "superseded": [], "conflicts": []})

outcomes = {}
for order in (("desk", "vrm"), ("vrm", "desk")):
    m = build_memory(f"nb05_coord_{order[0]}", fresh=True); rows_ = []
    for ch in order:
        write_pref(m, "email only" if ch == "desk" else "phone after 4pm", ch, "bank" if ch == "desk" else "agent", f"{ch}:same-day", rows_)
    outcomes[" then ".join(order)] = [r["value"] for r in view(vault_rows(m, "cust_test"), today=TODAY) if r["current"]]
print("current value by write order:", outcomes)

AUTHORITY = {"desk": 2, "contact_centre": 2, "vrm": 1}
def resolve_same_day(a: dict, b: dict) -> str:
    if a["observed_on"] != b["observed_on"]:
        return "later event date wins (supersede)"
    return "same day, different channels: keep both, mark as conflict, ask the customer" if a["channel"] != b["channel"] else "same channel: later write wins"
print("tie-break:", resolve_same_day({"observed_on": "2026-09-27", "channel": "desk"}, {"observed_on": "2026-09-27", "channel": "vrm"}))

Model us.anthropic.claude-haiku-4-5-20251001-v1:0 may not be available in region us-east-1
The 'faiss' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).
Model us.anthropic.claude-haiku-4-5-20251001-v1:0 may not be available in region us-east-1
The 'faiss' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).


current value by write order: {'desk then vrm': ['phone after 4pm'], 'vrm then desk': ['email only']}
tie-break: same day, different channels: keep both, mark as conflict, ask the customer


**Measure.** If the two orders give different current values, the store is order-dependent, which in a queue-based ingestion (SQS, notebook 04) means nondeterministic. **Watch out.** Event times on the same day need a finer clock (timestamps, not dates) and an explicit rule; "the banker outranks the VRM" is a policy decision, and for a *preference* the customer is the authority, so the safest rule is to ask.

## Recipe 19 · Which benchmarks test what

> ⚪ **Lower importance for this use case.** Orientation for choosing external benchmarks; the bank's own dataset remains the release criterion.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§VIII-B and Table I**: seven benchmark scenarios (Interaction, Personalization, Web, LongContext, Continual, Environments, Tool/Gen) and the benchmarks in each, for example **[94] LoCoMo**, **[95] LongMemEval**, **[128] HaluMem** and **[129] MemoryBench**. It also critiques them: interaction benchmarks rarely supervise conflicting updates, and personalization benchmarks assume clear supervision of what to store. *Used here:* the table and the gaps.
- *This project, not from a paper:* the mapping of each scenario to this use case, and what this series' dataset adds and lacks.

| Scenario (paper) | Examples named in the survey | What it would test for this use case | Covered by this series' own dataset? |
|---|---|---|---|
| Interaction | LoCoMo, LongMemEval, MemoryAgentBench, MSC, DialSim, RealMem | Recall across sessions and channels | Yes, in miniature (notebook 03); run LongMemEval for scale |
| Personalization | PersonaMem, PrefEval, LOCCO, MemoryBank | Preferences that change; persona drift | Yes for updates; no for drift over months |
| LongContext | HotpotQA, 2WikiMultihopQA, MuSiQue (multi-hop); HaluMem (memory hallucination) | Multi-hop relationships; invented memories | Graph questions (notebook 03); grounding checks (notebook 03 level 1) |
| Continual | MemoryBench, LifelongAgentBench, StreamBench, Evo-Memory | Quality as memory grows and learns | No: the main gap |
| Tool/Gen | ToolBench, GAIA | Memory around tool calls (system lookups) | Partly (recipe 9) |
| Web, Environments | WebArena, ALFWorld, Ego4D | Not central to a banking vault | No |

What this series' dataset adds that the public benchmarks largely do not: conflicting facts with gold handling (system of record vs customer), access boundaries graded as their own categories, commitments with due dates and states, and lineage-based retrieval metrics. What it lacks: scale, drift over time, and real data. **Watch out.** Public benchmark scores say little about a regulated setting on their own; use them for regression and comparison, and keep the bank's own dataset as the release criterion.

## Recipe 20 · Is the result stable?

> 🔴 **High importance for this use case.** A release decision taken on one run is taken on noise; every gate result depends on this.

**Problem.** Across this series' complete runs, vault accuracy moved between about 65% and 88% with no code change. A gate decided on one run is decided by noise.

**Sources and what each contributes** (reference numbers are the survey's; contributions as the survey describes them)
- **§VIII, synthesis of evaluation landscapes**: measure the specific impact of memory through ablation tests, and evaluate how agents handle changing information. No specific paper is cited for run-to-run stability. *Used here:* the motivation.
- *This project, not from a paper:* answering a frozen vault three times, per-question agreement, and the mean with its range.

**Recipe.** Freeze the vault (no re-extraction), answer the same questions three times, grade each answer, and report per-question agreement and the mean with its range.

In [23]:
# ──────────────────────────────────────────────────────────────────────────────────────────────────
# Sources in this cell
#   §VIII evaluation synthesis ->  isolate the impact of memory; evaluate under changing information
#   This project ->  repeated runs on a frozen vault, per-question stability, mean and range
# ──────────────────────────────────────────────────────────────────────────────────────────────────
subset = [q for q in vd["questions"] if q["id"] in ("v4", "v8", "v13", "v17", "v18", "v19", "v20")]
runs = []
for run in range(3):
    answers = run_parallel(lambda q: answer_vault(q, service, principals, vd)["answer"], subset)
    runs.append([judge_vault_answer(q, a).verdict == "CORRECT" for q, a in zip(subset, answers)])
table = pd.DataFrame({f"run {i + 1}": r for i, r in enumerate(runs)}, index=[q["id"] for q in subset])
table["stable"] = table.nunique(axis=1) == 1
print(table.replace({True: "ok", False: "WRONG"}).to_string())
per_run = [sum(r) / len(r) for r in runs]
print(f"\naccuracy per run: {[f'{a:.0%}' for a in per_run]}; mean {sum(per_run) / 3:.0%}, range {min(per_run):.0%} to {max(per_run):.0%}; "
      f"questions whose verdict flipped: {list(table.index[~table.stable])}")

     run 1  run 2  run 3 stable
v4   WRONG  WRONG  WRONG     ok
v8      ok  WRONG  WRONG  WRONG
v13  WRONG  WRONG  WRONG     ok
v17     ok     ok     ok     ok
v18     ok     ok     ok     ok
v19     ok     ok     ok     ok
v20  WRONG     ok  WRONG  WRONG

accuracy per run: ['57%', '57%', '43%']; mean 52%, range 43% to 57%; questions whose verdict flipped: ['v8', 'v20']


**Measure.** A flipping question is not evidence either way; gate on questions that are stable, or on the mean over runs with a stated range. Because the vault was frozen here, every flip is answering-and-judging variance; a full rebuild adds extraction variance on top. **Watch out.** Three runs is the minimum to see variance, not to estimate it well; for a release decision, use more runs or more questions per category.

# Questions and answers

Answer each question yourself first, then open it.

<details><summary><b>1. The survey calls a vector store a "degenerate graph". What does that mean, and what does it imply for the vault?</b></summary>

A vector store connects every memory to every other by similarity weight and nothing else, so it can only answer "what is similar to this?". A graph adds typed, directed relations (owns, director of, secured by), which is what multi-hop and relationship questions need. For the vault it means flat retrieval is the baseline, and explicit edges are added where the questions are about structure (recipe 6 routes between them).
</details>

<details><summary><b>2. Which life-cycle stage is the vault weakest at, by the survey's framework, and why does it matter for a bank?</b></summary>

Evolution: consolidating near-duplicates, reorganising and forgetting by significance, inferring links, and learning from outcomes (recipes 10 to 14). Extraction, storage and retrieval are governed and measured; evolution changes memory without a new interaction, so every evolution step needs the same lineage, policy and human approval as a write.
</details>

<details><summary><b>3. Why is valid time alone not enough for audit questions?</b></summary>

Valid time says when something was true in the world; audit asks what the bank knew when it acted. A resignation keyed in ten days late means the records showed two directors for ten days after the fact (recipe 3). Without transaction time you cannot reconstruct what an agent or banker saw.
</details>

<details><summary><b>4. When should the service traverse the graph, and when not?</b></summary>

When the question is about relationships (ownership, directorships, related entities, guarantors), the graph adds evidence that similarity cannot reach. For other questions it mostly adds tokens. A router decides; its misses are logged and audited (recipe 6).
</details>

<details><summary><b>5. A commitment involves the bank, the customer, an accountant, a due date and a reference. Why model it as a hyperedge?</b></summary>

Because the participants belong to one promise. Split into pairwise triples, a query for "everything involving Deepak" cannot tell which due date belongs to which promise; a hyperedge returns the whole commitment from any member (recipe 5).
</details>

<details><summary><b>6. Multi-round retrieval with a sufficiency check sounds strictly better. What does it cost, and what can it not fix?</b></summary>

Two or three model calls per round, more latency, and its own variance. It cannot find what retrieval never surfaces: a sufficiency checker does not know an overdue commitment exists unless a structured tool (open commitments, as-of views) is in the loop (recipe 8).
</details>

<details><summary><b>7. The CRM says 14 trucks; memory says the customer said 16. An agent also has a live CRM lookup. What should it say?</b></summary>

Both, with sources and dates, the CRM marked authoritative for the fact and the customer's statement as unverified (recipe 9). The live lookup is cited, not written back into memory as verified.
</details>

<details><summary><b>8. Two memories embed at 0.92 similarity. Can they be merged?</b></summary>

Only if they state the same fact. Commitments with different due dates, facts with different values, or facts about different customers can embed very close together. A model gate decides, merges are marked rather than deleted, and the merged record's source counts as a reconfirmation (recipe 10).
</details>

<details><summary><b>9. Can significance scores decide what to delete?</b></summary>

No. They rank what a person should review first. Retention in a bank is governed by a records schedule and legal holds; commitments, verified facts and open items are never candidates (recipe 11).
</details>

<details><summary><b>10. An inferred edge says Priya has an interest in the depot. Who may see it?</b></summary>

Only a principal who may read every premise it was inferred from, and only labelled as an inference with its derivation and confidence. It must never reach a customer channel as a fact, and it is not a beneficial-ownership finding (recipe 12).
</details>

<details><summary><b>11. A model proposes a new servicing playbook from a customer's history. What has to happen before an agent uses it?</b></summary>

A person with the authority approves it, with an owner and a review date, and invented evidence ids are rejected automatically. A learned playbook changes how every customer is treated, so it is a policy change (recipe 13).
</details>

<details><summary><b>12. What makes active inquiry safe in a customer channel?</b></summary>

Deterministic gap detection, at most one question per contact, never asking for something already on file (check retrieval first), never asking for sensitive values, a purpose the customer agreed to, and answers recorded as customer-stated (recipe 14).
</details>

<details><summary><b>13. Name the four dimensions of intrinsic graph quality and one metric for each.</b></summary>

Structural: entities named in records but missing from the graph. Semantic: share of edges supported by their source record. Temporal: functional predicates violated at the current time. Operational: build and traversal latency (recipe 15).
</details>

<details><summary><b>14. The VRM never reads the property company's records. How can the graph still leak information about it?</b></summary>

Through structure: permitted edges can name another customer's entity and its relationships, and inference over edges can reveal more than any single record. Control it with per-channel predicate allowlists and by giving inferred edges the most restrictive audience of their premises (recipes 16 and 12).
</details>

<details><summary><b>15. A customer tells the VRM that another bank customer is a director of her company. What happens to that claim?</b></summary>

It is stored as what she said (customer-stated, unverified), so the record is accurate. As structure, a governed predicate from an unverified source, or an edge reaching another customer's entity, is quarantined from traversal and queued for review (recipe 17).
</details>

<details><summary><b>16. Why is "the later write wins" unsafe when several agents write to one memory?</b></summary>

Because write order depends on queue timing, not on what happened, so the current value becomes nondeterministic. Order by event time with a fine-grained clock, and for same-time conflicts across channels keep both and resolve by an explicit rule or by asking (recipe 18).
</details>

<details><summary><b>17. Which public benchmarks would you add to the bank's own dataset, and what would they still not tell you?</b></summary>

LongMemEval or LoCoMo for cross-session recall at scale, HotpotQA-style multi-hop sets for relationship reasoning, HaluMem for invented memories, and a continual benchmark (MemoryBench, StreamBench) for drift. They still would not test access boundaries, system-of-record conflicts, commitments with due dates, or the bank's own data (recipe 19).
</details>

<details><summary><b>18. A question passes in one run and fails in the next with no change. What do you report?</b></summary>

That its verdict is unstable, with the per-run results. Gate on stable questions or on a mean with its range across several runs of a frozen vault, and separate extraction variance (rebuild) from answering and judging variance (recipe 20).
</details>

<details><summary><b>19. Why should "in July" be stored as a month rather than as 1 July, and which paper's idea is that?</b></summary>

Because the customer did not say 1 July: storing a day invents precision, and later arithmetic or ordering checks then treat a guess as a fact. It is part of TReMu's decoupling of mention time from event time [37]; the granularity field and the "return None rather than guess" rule are this series' governed version (recipe 2, `ground_time`).
</details>

<details><summary><b>20. What does a role-labelled hyperedge add over a plain set of members?</b></summary>

It says who promised, who benefits, who must be copied and what is being delivered, each with its source, so "what do we owe that involves the accountant?" returns the overdue promise in which he is the person to copy, and records that are not bank promises to the customer are filtered out. HyperGraphRAG [39] contributes the n-ary hyperedge; the role labels are the reified form used in property graphs (recipe 5).
</details>